# CallVerse — Qwen3-VL-2B QLoRA for visual evidence (ShopNova + Atlas Travel)

**What this notebook does:** it builds the 150-photo CallVerse vision dataset from its public sources,
measures the original Qwen3-VL-2B-Instruct, fine-tunes it with QLoRA (4-bit NF4 + LoRA), selects the best
checkpoint on the **validation** split only, compares the 3 systems on the held-out **test** split, and packs
everything into `callverse_vision_qlora_results.zip`.

**How to run on Kaggle:** Settings → Accelerator **GPU T4 x2** (or P100) → Settings → **Internet on** → **Run All**.
When it finishes, download `/kaggle/working/callverse_vision_qlora_results.zip` from the Output panel.

**What the model learns:** only what is *visible* in a photo (visual evidence). It never sees the customer's text
and never outputs intent, emotion, refund eligibility or any business decision — Intake (text) and the
Coordinator (rules) do that. The photos are all real; the customer messages in the dataset are written examples
used later for the late-fusion tests in CallVerse, not for this training.

## Cell 1 — Environment and GPU detection

In [ ]:
import json, os, platform, subprocess, sys, time
T_START = time.time()
WORK = "/kaggle/working" if os.path.isdir("/kaggle/working") else os.path.abspath("work")
os.makedirs(WORK, exist_ok=True)
os.chdir(WORK)
import torch
if not torch.cuda.is_available():
    raise SystemExit("No GPU found. In Kaggle: Settings -> Accelerator -> GPU T4 x2 (or P100), then Run All again.")
dev = torch.cuda.get_device_properties(0)
CAP = torch.cuda.get_device_capability(0)
BF16 = torch.cuda.is_bf16_supported() and CAP[0] >= 8
env = {
    "python": sys.version.split()[0], "platform": platform.platform(),
    "torch": torch.__version__, "cuda_runtime": torch.version.cuda, "cudnn": torch.backends.cudnn.version(),
    "gpu_count": torch.cuda.device_count(), "gpu": dev.name, "gpu_vram_gb": round(dev.total_memory / 2**30, 1),
    "compute_capability": f"{CAP[0]}.{CAP[1]}", "bf16_supported": BF16, "fp16_supported": True,
}
try:
    env["nvidia_smi"] = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total,driver_version",
                                        "--format=csv,noheader"], capture_output=True, text=True).stdout.strip()
except Exception as e:
    env["nvidia_smi"] = f"unavailable: {e}"
print(json.dumps(env, indent=1))

## Cell 2 — Dependencies
Kaggle's own PyTorch / torchvision (built for its CUDA) are kept. Qwen3-VL needs transformers ≥ 4.57; the pinned
versions below are the ones the code was checked with (CPU check, see README). If Kaggle's torch is older than 2.6,
transformers 5 cannot run and the cell falls back to transformers 4.57.x.

In [ ]:
from packaging.version import Version
TORCH_OK_FOR_TF5 = Version(torch.__version__.split("+")[0]) >= Version("2.6")
tf_spec = "transformers==5.22.1" if TORCH_OK_FOR_TF5 else "transformers>=4.57.1,<4.58"
pkgs = [tf_spec, "peft==0.21.2", "accelerate==1.15.0", "bitsandbytes==0.50.2", "pyarrow", "scikit-learn", "matplotlib"]
print("installing:", pkgs)
r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", *pkgs], capture_output=True, text=True)
print(r.stdout[-2000:], r.stderr[-3000:])
import transformers, peft, accelerate   # first import happens here, after the install
from transformers import AutoProcessor, Qwen3VLForConditionalGeneration, BitsAndBytesConfig
try:
    import bitsandbytes as bnb
    BNB_VERSION = bnb.__version__
except Exception as e:
    BNB_VERSION = f"import failed: {e}"
env.update({"transformers": transformers.__version__, "peft": peft.__version__, "accelerate": accelerate.__version__,
            "bitsandbytes": BNB_VERSION})
print({k: env[k] for k in ("torch", "transformers", "peft", "accelerate", "bitsandbytes")})

### Shared code (written from the repository files `cv_vision_lib.py` and `build_dataset.py`)

In [ ]:
%%writefile cv_vision_lib.py
"""CallVerse vision: prompts, data encoding, evaluation and QLoRA training for Qwen3-VL-2B.

One file, used by the Kaggle notebook (copied in with %%writefile) and by the local CPU checks.

Design rules (CLAUDE.md + task):
- The vision model sees ONLY the photo. It never sees the customer's words: it reports what is VISIBLE
  (visual evidence). The customer's intent comes from Intake (text). They are joined later (late fusion).
- The answer is one JSON object with fixed keys. No intent, no refund decision, no emotion, no business facts.
- Loss is computed only on the answer tokens (the prompt and the image tokens are masked with -100).
"""
import json
import math
import os
import random
import re
import time

ISSUE_TYPES = ["cracked_screen", "screen_display_defect", "damaged_packaging", "damaged_luggage",
               "dirty_or_messy_room", "damaged_room_fixture", "no_visible_issue", "unclear"]
DAMAGE_TYPES = ISSUE_TYPES[:6]
KEYS = ["visible_damage", "object", "issue_type", "description", "confidence_level", "requires_review"]

# Mixed precision used for EVERY forward pass (train, eval loss, generation): None = off (CPU / fp32).
# The notebook sets it to torch.bfloat16 or torch.float16 after checking the GPU.
AMP_DTYPE = None


def _autocast():
    import contextlib
    import torch
    if AMP_DTYPE is None or not torch.cuda.is_available():
        return contextlib.nullcontext()
    return torch.autocast("cuda", dtype=AMP_DTYPE)

# Prompt 1 = the original style used in Molka's benchmark (short, one question) + our output keys.
PROMPT_BASE = (
    "Look carefully at this image sent by a customer. Is there clearly visible physical damage? "
    "Return only JSON with the keys visible_damage (true, false or null), object, issue_type, description, "
    "confidence_level and requires_review."
)

# Prompt 2 = improved prompt: allowed values, grounding rules, uncertainty. Used for the base model AND for
# fine-tuning, so the "prompt only" and "fine-tuned" rows of the benchmark differ only by the training.
PROMPT_IMPROVED = (
    "You check photos sent to a customer-service centre (an online shop and a travel agency).\n"
    "Describe ONLY what is visible in the photo. Do not guess the customer's wishes, feelings, identity, "
    "orders, bookings, payments or any decision.\n"
    "Return one JSON object and nothing else, with exactly these keys:\n"
    '{"visible_damage": true | false | null, "object": "<main object, snake_case>", '
    '"issue_type": "<one value from the list>", "description": "<one short sentence about what is visible>", '
    '"confidence_level": "high" | "medium" | "low", "requires_review": true | false}\n'
    "issue_type values: cracked_screen, screen_display_defect, damaged_packaging, damaged_luggage, "
    "dirty_or_messy_room, damaged_room_fixture, no_visible_issue, unclear.\n"
    "Rules: visible_damage is true only for clearly visible damage or dirt; false when the object looks intact "
    "or clean (issue_type no_visible_issue); null when the photo does not allow a decision (issue_type unclear, "
    "confidence_level low). requires_review is true whenever visible_damage is not false or confidence is not high."
)


def answer_text(target):
    """Canonical answer string (fixed key order) the model learns to write."""
    return json.dumps({k: target[k] for k in KEYS}, ensure_ascii=False)


def load_rows(dataset_dir):
    rows = [json.loads(l) for l in open(os.path.join(dataset_dir, "dataset.jsonl"), encoding="utf-8") if l.strip()]
    for r in rows:
        r["image_path"] = os.path.join(dataset_dir, r["image"])
    return rows


def open_image(path):
    from PIL import Image
    with Image.open(path) as im:
        return im.convert("RGB")


# ---------------- parsing ----------------
def parse_output(text):
    """Return (dict or None, reason). Valid = one JSON object with the 6 keys and allowed values."""
    text = text.strip()
    text = re.sub(r"^```(?:json)?\s*|\s*```$", "", text)
    m = re.search(r"\{.*\}", text, re.S)
    if not m:
        return None, "no_json"
    try:
        obj = json.loads(m.group(0))
    except json.JSONDecodeError:
        return None, "bad_json"
    if not isinstance(obj, dict) or set(obj) != set(KEYS):
        return None, "wrong_keys"
    if obj["visible_damage"] not in (True, False, None):
        return None, "bad_visible_damage"
    if obj["issue_type"] not in ISSUE_TYPES:
        return None, "bad_issue_type"
    if obj["confidence_level"] not in ("high", "medium", "low"):
        return None, "bad_confidence"
    if not isinstance(obj["requires_review"], bool) or not isinstance(obj["description"], str):
        return None, "bad_types"
    return obj, "ok"


def lenient_damage(raw):
    """visible_damage read from ANY JSON-looking answer (even with wrong keys), like Molka's binary benchmark
    ({"damaged": true}). Only used for the 'lenient' damage recall, so the original prompt is not punished twice."""
    m = re.search(r'"(?:visible_damage|damaged|damage)"\s*:\s*(true|false|null)', raw, re.I)
    return {"true": True, "false": False}.get(m.group(1).lower()) if m else None


# ---------------- encoding ----------------
def chat(prompt, image, answer=None):
    msgs = [{"role": "user", "content": [{"type": "image", "image": image}, {"type": "text", "text": prompt}]}]
    if answer is not None:
        msgs.append({"role": "assistant", "content": [{"type": "text", "text": answer}]})
    return msgs


def encode_for_training(processor, image, prompt, answer):
    """Tokenise prompt+answer with the image; labels = -100 everywhere except the answer and its end token.
    The prompt text is the exact chat-template prefix, so its token count is the mask length."""
    prompt_text = processor.apply_chat_template(chat(prompt, image), tokenize=False, add_generation_prompt=True)
    full_text = prompt_text + answer + "<|im_end|>\n"
    full = processor(text=[full_text], images=[image], return_tensors="pt")
    prompt_only = processor(text=[prompt_text], images=[image], return_tensors="pt")
    n_prompt = prompt_only["input_ids"].shape[1]
    ids = full["input_ids"]
    if not (ids[0, :n_prompt] == prompt_only["input_ids"][0]).all():
        raise RuntimeError("prompt is not a prefix of prompt+answer: label mask would be wrong")
    labels = ids.clone()
    labels[:, :n_prompt] = -100
    full["labels"] = labels
    return full


def count_image_tokens(processor, input_ids):
    pad_id = processor.tokenizer.convert_tokens_to_ids("<|image_pad|>")
    return int((input_ids == pad_id).sum())


# ---------------- generation + evaluation ----------------
def generate(model, processor, image, prompt, max_new_tokens=160):
    import torch
    text = processor.apply_chat_template(chat(prompt, image), tokenize=False, add_generation_prompt=True)
    inputs = processor(text=[text], images=[image], return_tensors="pt").to(model.device)
    if torch.cuda.is_available():
        torch.cuda.synchronize()
    t0 = time.perf_counter()
    with torch.no_grad(), _autocast():
        out = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False, temperature=None,
                             top_p=None, top_k=None, use_cache=True)
    if torch.cuda.is_available():
        torch.cuda.synchronize()
    dt = time.perf_counter() - t0
    new = out[0, inputs["input_ids"].shape[1]:]
    return processor.decode(new, skip_special_tokens=True), dt


def predict(model, processor, rows, prompt, log=print, max_new_tokens=160):
    model.eval()   # no dropout while generating (train_run switches back to train mode itself)
    preds = []
    for i, r in enumerate(rows, 1):
        img = open_image(r["image_path"])
        raw, dt = generate(model, processor, img, prompt, max_new_tokens)
        obj, why = parse_output(raw)
        preds.append({"id": r["id"], "raw": raw, "parsed": obj, "parse": why, "latency_s": round(dt, 3)})
        if i % 10 == 0:
            log(f"    {i}/{len(rows)} predicted")
    return preds


def _prf(tp, fp, fn):
    p = tp / (tp + fp) if tp + fp else 0.0
    r = tp / (tp + fn) if tp + fn else 0.0
    f = 2 * p * r / (p + r) if p + r else 0.0
    return p, r, f


def point_metrics(gold, pred_issue, pred_damage, valid):
    """gold: list of targets. pred_issue: list (None = invalid). Invalid outputs count as wrong."""
    n = len(gold)
    labels = sorted({g["issue_type"] for g in gold})
    acc = sum(p == g["issue_type"] for p, g in zip(pred_issue, gold)) / n
    per = {}
    for c in labels:
        tp = sum(p == c and g["issue_type"] == c for p, g in zip(pred_issue, gold))
        fp = sum(p == c and g["issue_type"] != c for p, g in zip(pred_issue, gold))
        fn = sum(p != c and g["issue_type"] == c for p, g in zip(pred_issue, gold))
        pr, rc, f1 = _prf(tp, fp, fn)
        per[c] = {"precision": pr, "recall": rc, "f1": f1, "support": tp + fn}
    macro_p = sum(v["precision"] for v in per.values()) / len(per)
    macro_r = sum(v["recall"] for v in per.values()) / len(per)
    macro_f1 = sum(v["f1"] for v in per.values()) / len(per)
    # damage = binary task "is there visible damage/dirt?" (gold True vs gold False; gold null rows excluded)
    dmg = [(pd, g["visible_damage"]) for pd, g in zip(pred_damage, gold) if g["visible_damage"] is not None]
    tp = sum(pd is True and gd is True for pd, gd in dmg)
    fn = sum(pd is not True and gd is True for pd, gd in dmg)
    fp = sum(pd is True and gd is False for pd, gd in dmg)
    tn = sum(pd is not True and gd is False for pd, gd in dmg)
    d_p, d_r, d_f = _prf(tp, fp, fn)
    return {"n": n, "accuracy": acc, "macro_precision": macro_p, "macro_recall": macro_r, "macro_f1": macro_f1,
            "damage_recall": d_r, "damage_precision": d_p, "damage_f1": d_f,
            "false_alarm_rate": fp / (fp + tn) if fp + tn else 0.0,
            "json_validity": sum(valid) / n, "per_class": per,
            "damage_confusion": {"tp": tp, "fn": fn, "fp": fp, "tn": tn}}


def metrics(rows, preds, n_boot=2000, seed=42):
    """Point metrics + 95 % bootstrap intervals (resampling photos). Small test set -> wide intervals."""
    gold = [r["target"] for r in rows]
    pi = [p["parsed"]["issue_type"] if p["parsed"] else None for p in preds]
    pdm = [p["parsed"]["visible_damage"] if p["parsed"] else None for p in preds]
    valid = [p["parsed"] is not None for p in preds]
    m = point_metrics(gold, pi, pdm, valid)
    lenient = [lenient_damage(p["raw"]) for p in preds]
    dm = [(ld, g["visible_damage"]) for ld, g in zip(lenient, gold) if g["visible_damage"] is True]
    m["damage_recall_lenient"] = sum(ld is True for ld, _ in dm) / len(dm) if dm else 0.0
    rng = random.Random(seed)
    keys = ["accuracy", "macro_f1", "damage_recall", "json_validity", "false_alarm_rate"]
    boots = {k: [] for k in keys}
    n = len(gold)
    for _ in range(n_boot):
        idx = [rng.randrange(n) for _ in range(n)]
        b = point_metrics([gold[i] for i in idx], [pi[i] for i in idx], [pdm[i] for i in idx], [valid[i] for i in idx])
        for k in keys:
            boots[k].append(b[k])
    m["ci95"] = {k: [sorted(v)[int(0.025 * n_boot)], sorted(v)[int(0.975 * n_boot) - 1]] for k, v in boots.items()}
    lat = sorted(p["latency_s"] for p in preds)
    m["latency_s"] = {"mean": sum(lat) / len(lat), "median": lat[len(lat) // 2],
                      "p95": lat[min(len(lat) - 1, math.ceil(0.95 * len(lat)) - 1)]}
    return m


def paired_bootstrap(rows, preds_a, preds_b, key="issue_type", n_boot=2000, seed=7):
    """P(B better than A) on accuracy by paired resampling of the SAME photos (difference + 95 % interval)."""
    gold = [r["target"][key] for r in rows]
    a = [int(bool(p["parsed"]) and p["parsed"][key] == g) for p, g in zip(preds_a, gold)]
    b = [int(bool(p["parsed"]) and p["parsed"][key] == g) for p, g in zip(preds_b, gold)]
    rng = random.Random(seed)
    n = len(gold)
    diffs = []
    for _ in range(n_boot):
        idx = [rng.randrange(n) for _ in range(n)]
        diffs.append(sum(b[i] - a[i] for i in idx) / n)
    diffs.sort()
    return {"mean_diff": sum(b[i] - a[i] for i in range(n)) / n, "ci95": [diffs[int(0.025 * n_boot)], diffs[int(0.975 * n_boot) - 1]],
            "p_b_better": sum(d > 0 for d in diffs) / n_boot}


# ---------------- training ----------------
def lora_target_regex():
    """LoRA only on the LANGUAGE model linear layers. The vision encoder stays frozen (small data: less
    over-fitting, less memory). The image still drives the loss: the LLM layers read the image tokens."""
    return r".*language_model.*\.(q_proj|k_proj|v_proj|o_proj|gate_proj|up_proj|down_proj)$"


def eval_loss(model, processor, rows, prompt, shuffle_images=False, seed=0):
    """Mean answer-token loss. shuffle_images=True pairs each answer with ANOTHER photo: if the model really
    uses the image, this loss must be clearly higher (image-conditioning check)."""
    import torch
    imgs = [r["image_path"] for r in rows]
    if shuffle_images:
        rng = random.Random(seed)
        imgs = imgs[:]
        while True:
            rng.shuffle(imgs)
            if all(a != r["image_path"] for a, r in zip(imgs, rows)) or len(rows) < 2:
                break
    model.eval()
    tot, cnt = 0.0, 0
    with torch.no_grad():
        for r, ip in zip(rows, imgs):
            batch = encode_for_training(processor, open_image(ip), prompt, answer_text(r["target"])).to(model.device)
            n_tok = int((batch["labels"] != -100).sum())
            with _autocast():
                out = model(**batch)
            tot += float(out.loss) * n_tok
            cnt += n_tok
    return tot / max(cnt, 1)


def train_run(model, processor, train_rows, val_rows, cfg, out_dir, log=print, use_scaler=False, epoch_callback=None):
    """Plain PyTorch loop (no TRL): batch 1 + gradient accumulation, cosine schedule with warm-up,
    answer-only loss, val loss every epoch, adapter saved every epoch, resume from the last saved epoch.
    epoch_callback(epoch, adapter_dir) -> dict of extra validation numbers (e.g. val macro-F1 by generation).
    Early stop: cfg["patience"] epochs in a row without a better val loss."""
    import torch
    from transformers import get_cosine_schedule_with_warmup

    os.makedirs(out_dir, exist_ok=True)
    params = [p for p in model.parameters() if p.requires_grad]
    opt = torch.optim.AdamW(params, lr=cfg["lr"], weight_decay=0.0)
    steps_per_epoch = math.ceil(len(train_rows) / cfg["grad_accum"])
    total = steps_per_epoch * cfg["epochs"]
    sched = get_cosine_schedule_with_warmup(opt, max(1, int(0.1 * total)), total)
    scaler = torch.amp.GradScaler("cuda") if use_scaler else None
    history, start_epoch = [], 0
    state_file = os.path.join(out_dir, "state.pt")
    if os.path.exists(state_file):   # recovery after an interruption
        st = torch.load(state_file, map_location="cpu", weights_only=False)
        from peft import set_peft_model_state_dict
        set_peft_model_state_dict(model, st["adapter"])
        opt.load_state_dict(st["opt"])
        sched.load_state_dict(st["sched"])
        history, start_epoch = st["history"], st["epoch"] + 1
        log(f"  resumed {out_dir} after epoch {st['epoch'] + 1}")
    rng = random.Random(cfg["seed"])
    order = list(range(len(train_rows)))
    for epoch in range(start_epoch, cfg["epochs"]):
        model.train()
        rng_epoch = random.Random(cfg["seed"] * 1000 + epoch)
        rng_epoch.shuffle(order)
        run_loss, run_n, t0 = 0.0, 0, time.time()
        opt.zero_grad(set_to_none=True)
        for k, idx in enumerate(order, 1):
            r = train_rows[idx]
            batch = encode_for_training(processor, open_image(r["image_path"]), cfg["prompt"],
                                        answer_text(r["target"])).to(model.device)
            with _autocast():
                loss = model(**batch).loss / cfg["grad_accum"]
            if not torch.isfinite(loss):
                raise FloatingPointError(f"non-finite loss at epoch {epoch + 1}, example {r['id']}")
            (scaler.scale(loss) if scaler else loss).backward()
            run_loss += float(loss) * cfg["grad_accum"]
            run_n += 1
            if k % cfg["grad_accum"] == 0 or k == len(order):
                if scaler:
                    scaler.unscale_(opt)
                gn = torch.nn.utils.clip_grad_norm_(params, 1.0)
                if scaler:
                    scaler.step(opt)
                    scaler.update()
                else:
                    opt.step()
                sched.step()
                opt.zero_grad(set_to_none=True)
        tr = run_loss / run_n
        vl = eval_loss(model, processor, val_rows, cfg["prompt"])
        mem = torch.cuda.max_memory_allocated() / 2**30 if torch.cuda.is_available() else 0.0
        rec = {"epoch": epoch + 1, "train_loss": tr, "val_loss": vl, "last_grad_norm": float(gn),
               "lr": sched.get_last_lr()[0], "seconds": round(time.time() - t0, 1), "max_gpu_mem_gb": round(mem, 2)}
        log(f"  epoch {epoch + 1}: train_loss {tr:.4f}  val_loss {vl:.4f}  grad_norm {float(gn):.3f}  "
            f"{rec['seconds']} s  peak GPU {rec['max_gpu_mem_gb']} GB")
        ep_dir = os.path.join(out_dir, f"epoch_{epoch + 1}")
        model.save_pretrained(ep_dir)
        if epoch_callback:
            rec.update(epoch_callback(epoch + 1, ep_dir))
            model.train()
        history.append(rec)
        from peft import get_peft_model_state_dict
        torch.save({"adapter": get_peft_model_state_dict(model), "opt": opt.state_dict(), "sched": sched.state_dict(),
                    "history": history, "epoch": epoch}, state_file)
        best_vl = min(h["val_loss"] for h in history)
        since_best = len(history) - 1 - [h["val_loss"] for h in history].index(best_vl)
        if cfg.get("patience") and since_best >= cfg["patience"]:
            log(f"  early stop: val loss has not improved for {since_best} epoch(s)")
            break
    return history


def load_adapter_weights(model, adapter_dir):
    """Put the LoRA weights saved in adapter_dir into the (same-shape) PEFT model."""
    from peft import set_peft_model_state_dict
    from safetensors.torch import load_file
    set_peft_model_state_dict(model, load_file(os.path.join(adapter_dir, "adapter_model.safetensors")))

In [ ]:
%%writefile build_dataset.py
"""Build the CallVerse vision dataset (150 real photos) from its public sources.

Why a builder and not a zip in git? The repository is public and some sources do not allow redistribution
(see DATASET_PROVENANCE.md). So git keeps only the MANIFEST (where each photo comes from + its sha256 + the
labels), and this script downloads the photos again from the pinned source revisions. The same code runs on a
PC and inside the Kaggle notebook.

Steps (each one is a check from the task):
  1. download each source once (pinned revision) into a cache folder
  2. take the exact photo, check its sha256 against the manifest (integrity: same bytes as reviewed)
  3. normalise it: EXIF rotation, RGB, longest side <= 1024 px, JPEG quality 92
  4. quality checks: decodable, size, blur, brightness -> doubtful photos go to the review queue
  5. exact duplicates (sha256) and near duplicates (64-bit difference hash) -> merged into one group
  6. label checks (allowed values, damage flag consistent with the issue type, reason allowed for the company)
  7. stratified GROUP split 70/15/15 (seed 42): photos of the same suitcase / box / hotel / room stay together
  8. write dataset.jsonl, dataset_summary.json, review_queue.csv

Usage:  python build_dataset.py --manifest manifest.jsonl --out callverse_vision_dataset [--cache cache]
Only standard library + Pillow + numpy (+ pyarrow for the two parquet sources).
"""
import argparse, collections, csv, hashlib, io, json, os, random, sys, urllib.request, zipfile

from PIL import Image, ImageOps

# Visual evidence classes (what can be SEEN). Customer intent is NOT a visual class.
DAMAGE_TYPES = ["cracked_screen", "screen_display_defect", "damaged_packaging", "damaged_luggage",
                "dirty_or_messy_room", "damaged_room_fixture"]
ISSUE_TYPES = DAMAGE_TYPES + ["no_visible_issue", "unclear"]
CONFIDENCE = ["high", "medium", "low"]
# Contact reasons per company (verified on main, config.yaml companies: on 2026-11-01).
REASONS = {
    "shopnova": ["order_status", "late_delivery", "refund_request", "cancel_order", "return_item", "damaged_item", "other"],
    "atlas": ["booking_status", "delay_disruption", "cancellation", "refund_request", "compensation_request",
              "rebooking", "special_assistance", "other"],
}
MAX_SIDE = 1024
SEED = 42
SPLIT_SHARES = {"test": 0.15, "val": 0.15}   # the rest is train (~70 %)


def target_for(row):
    """The structured answer the model must learn. Every field is visible in the photo (or says it is not)."""
    issue = row["issue_type"]
    visible = True if issue in DAMAGE_TYPES else (False if issue == "no_visible_issue" else None)
    return {
        "visible_damage": visible,                 # None = the photo does not allow a decision
        "object": row["object"],
        "issue_type": issue,
        "description": row["description"],
        "confidence_level": row["confidence_level"],
        # A human (or the Coordinator's rules) must look at any damage claim or any doubtful photo.
        "requires_review": visible is not False or row["confidence_level"] != "high",
    }


# ---------- 1. download sources ----------
def _get(url, dest):
    if os.path.exists(dest) and os.path.getsize(dest) > 0:
        return dest
    os.makedirs(os.path.dirname(dest), exist_ok=True)
    tmp = dest + ".part"
    req = urllib.request.Request(url, headers={"User-Agent": "callverse-dataset-builder"})
    with urllib.request.urlopen(req, timeout=600) as r, open(tmp, "wb") as f:
        while True:
            chunk = r.read(1 << 20)
            if not chunk:
                break
            f.write(chunk)
    os.replace(tmp, dest)
    return dest


def hf_url(repo, rev, path):
    return f"https://huggingface.co/datasets/{repo}/resolve/{rev}/{urllib.request.quote(path)}"


class Sources:
    """Gives the original bytes of one manifest row, downloading (and caching) what is needed."""

    def __init__(self, cache):
        self.cache = cache
        self._parquet = {}
        self._zip = {}

    def _cached(self, *parts):
        return os.path.join(self.cache, *[p.replace("/", "__") for p in parts])

    def raw_bytes(self, row):
        kind, repo, rev, ref = row["source_kind"], row["source_repo"], row["source_revision"], row["source_ref"]
        if kind == "hf":
            return open(_get(hf_url(repo, rev, ref["path"]), self._cached(repo, rev, ref["path"])), "rb").read()
        if kind == "github":
            url = f"https://raw.githubusercontent.com/{repo}/{rev}/{urllib.request.quote(ref['path'])}"
            return open(_get(url, self._cached(repo, rev, ref["path"])), "rb").read()
        if kind == "hf_parquet":
            import pyarrow.parquet as pq
            key = (repo, ref["parquet"])
            if key not in self._parquet:
                local = _get(hf_url(repo, rev, ref["parquet"]), self._cached(repo, rev, ref["parquet"]))
                self._parquet[key] = pq.ParquetFile(local)
            pf = self._parquet[key]
            if "row" in ref:
                # read only the row group that holds this row (the files are ~500 MB)
                start = 0
                for g in range(pf.num_row_groups):
                    n = pf.metadata.row_group(g).num_rows
                    if ref["row"] < start + n:
                        col = pf.read_row_group(g, columns=["image"]).column("image")
                        return col[ref["row"] - start].as_py()["bytes"]
                    start += n
                raise KeyError(f"row {ref['row']} not in {ref['parquet']}")
            cache_key = key + (ref["key_column"],)
            if cache_key not in self._parquet:
                t = pf.read(columns=[ref["key_column"]]).column(0).to_pylist()
                self._parquet[cache_key] = {k: i for i, k in enumerate(t)}
            i = self._parquet[cache_key][ref["key"]]
            return pf.read(columns=["image"]).column("image")[i].as_py()["bytes"]
        if kind == "hf_zip":
            key = (repo, ref["zip"])
            if key not in self._zip:
                local = _get(hf_url(repo, rev, ref["zip"]), self._cached(repo, rev, ref["zip"]))
                self._zip[key] = zipfile.ZipFile(local)
            return self._zip[key].read(ref["member"])
        raise ValueError(f"unknown source kind {kind}")


# ---------- 3-5. image checks ----------
def normalise(raw):
    with Image.open(io.BytesIO(raw)) as im:
        im.load()
        im = ImageOps.exif_transpose(im).convert("RGB")
    im.thumbnail((MAX_SIDE, MAX_SIDE), Image.LANCZOS)
    buf = io.BytesIO()
    im.save(buf, format="JPEG", quality=92)
    return im, buf.getvalue()


def quality(im):
    """Return (blur score, brightness 0-255). Blur = variance of a Laplacian on a 256-px grey copy."""
    import numpy as np
    g = im.convert("L").copy()
    g.thumbnail((256, 256))
    a = np.asarray(g, dtype=np.float32)
    lap = a[1:-1, :-2] + a[1:-1, 2:] + a[:-2, 1:-1] + a[2:, 1:-1] - 4 * a[1:-1, 1:-1]
    return float(lap.var()), float(a.mean())


def dhash(im, size=8):
    """64-bit difference hash: tiny grey image, compare each pixel with its right neighbour."""
    g = im.convert("L").resize((size + 1, size), Image.LANCZOS)
    px = list(g.getdata())
    bits = 0
    for y in range(size):
        for x in range(size):
            bits = (bits << 1) | (px[y * (size + 1) + x] > px[y * (size + 1) + x + 1])
    return bits


def hamming(a, b):
    return bin(a ^ b).count("1")


# ---------- 6. labels ----------
def check_labels(rows):
    problems = []
    for r in rows:
        if r["issue_type"] not in ISSUE_TYPES:
            problems.append((r["id"], f"unknown issue_type {r['issue_type']}"))
        if r["confidence_level"] not in CONFIDENCE:
            problems.append((r["id"], f"unknown confidence {r['confidence_level']}"))
        if r["issue_type"] == "unclear" and r["confidence_level"] != "low":
            problems.append((r["id"], "unclear photos must have low confidence"))
        if r["company_reason"] is not None and r["company_reason"] not in REASONS[r["domain"]]:
            problems.append((r["id"], f"reason {r['company_reason']} not allowed for {r['domain']}"))
        if (r["company_reason"] is None) != (r["customer_message"] is None):
            problems.append((r["id"], "reason and message must both be set or both be empty"))
        if not r["description"].strip() or not r["object"].strip():
            problems.append((r["id"], "empty description or object"))
    return problems


# ---------- 7. split ----------
def group_split(rows, seed=SEED):
    """Stratified group split. Groups are visited class by class (rarest class first) and placed in test,
    then val, until each split holds its share of that class; the rest goes to train. Deterministic."""
    groups = collections.defaultdict(list)
    for r in rows:
        groups[r["group"]].append(r)
    cls_total = collections.Counter(r["issue_type"] for r in rows)
    have = {s: collections.Counter() for s in ("test", "val", "train")}
    assigned = {}
    rng = random.Random(seed)
    for cls in sorted(cls_total, key=lambda c: (cls_total[c], c)):
        gids = sorted({r["group"] for r in rows if r["issue_type"] == cls} - set(assigned))
        rng.shuffle(gids)
        for g in gids:
            n_cls = sum(r["issue_type"] == cls for r in groups[g])
            for s in ("test", "val"):
                want = max(1, round(cls_total[cls] * SPLIT_SHARES[s]))
                if have[s][cls] + n_cls <= want or (have[s][cls] == 0 and cls_total[cls] >= 3):
                    break
            else:
                s = "train"
            assigned[g] = s
            for r in groups[g]:
                have[s][r["issue_type"]] += 1
    for r in rows:
        r["split"] = assigned[r["group"]]
    return have


def build(manifest, out, cache, log=print):
    rows = [json.loads(l) for l in open(manifest, encoding="utf-8") if l.strip()]
    os.makedirs(os.path.join(out, "images"), exist_ok=True)
    src = Sources(cache)
    review, hashes = [], {}
    for i, r in enumerate(rows, 1):
        raw = src.raw_bytes(r)
        got = hashlib.sha256(raw).hexdigest()
        if got != r["sha256_source"]:
            raise RuntimeError(f"{r['id']}: source bytes changed (sha256 {got[:12]} != {r['sha256_source'][:12]})")
        im, jpg = normalise(raw)
        r["image"] = f"images/{r['id']}.jpg"
        with open(os.path.join(out, r["image"]), "wb") as f:
            f.write(jpg)
        blur, bright = quality(im)
        r["_q"] = dict(width=im.width, height=im.height, blur=round(blur, 1), brightness=round(bright, 1))
        if min(im.size) < 256:
            review.append((r["id"], "short side under 256 px"))
        if blur < 60:
            review.append((r["id"], f"low sharpness (blur score {blur:.0f})"))
        if bright < 45 or bright > 235:
            review.append((r["id"], f"very dark or very bright (mean {bright:.0f})"))
        if r["confidence_level"] == "low":
            review.append((r["id"], "label is 'unclear' / low confidence by design"))
        r["_dh"] = dhash(im)
        h = hashlib.sha256(jpg).hexdigest()
        if h in hashes:
            raise RuntimeError(f"exact duplicate: {r['id']} == {hashes[h]}")
        hashes[h] = r["id"]
        if i % 25 == 0:
            log(f"  {i}/{len(rows)} photos ready")
    # near duplicates -> same group (union-find), so they can never sit on both sides of a split
    parent = {r["group"]: r["group"] for r in rows}

    def find(g):
        while parent[g] != g:
            parent[g] = parent[parent[g]]
            g = parent[g]
        return g

    near = []
    for a in range(len(rows)):
        for b in range(a + 1, len(rows)):
            d = hamming(rows[a]["_dh"], rows[b]["_dh"])
            if d <= 6:
                near.append((rows[a]["id"], rows[b]["id"], d))
                parent[find(rows[a]["group"])] = find(rows[b]["group"])
    for r in rows:
        r["group"] = find(r["group"])
    for a, b, d in near:
        review.append((a, f"near duplicate of {b} (dHash distance {d}); kept in the same group"))
    problems = check_labels(rows)
    if problems:
        raise RuntimeError(f"label problems: {problems}")
    have = group_split(rows)
    # leakage check: no group in two splits
    seen = collections.defaultdict(set)
    for r in rows:
        seen[r["group"]].add(r["split"])
    leaks = [g for g, s in seen.items() if len(s) > 1]
    if leaks:
        raise RuntimeError(f"leakage: groups in several splits: {leaks}")
    with open(os.path.join(out, "dataset.jsonl"), "w", encoding="utf-8") as f:
        for r in rows:
            f.write(json.dumps({"id": r["id"], "image": r["image"], "domain": r["domain"],
                                "customer_message": r["customer_message"], "company_reason": r["company_reason"],
                                "group": r["group"], "split": r["split"], "target": target_for(r)},
                               ensure_ascii=False) + "\n")
    with open(os.path.join(out, "review_queue.csv"), "w", newline="", encoding="utf-8") as f:
        w = csv.writer(f)
        w.writerow(["id", "reason"])
        w.writerows(review)
    summary = {
        "images": len(rows),
        "real_photos": len(rows), "synthetic_images": 0,
        "by_split": dict(collections.Counter(r["split"] for r in rows)),
        "by_issue_type": dict(collections.Counter(r["issue_type"] for r in rows)),
        "by_issue_type_and_split": {s: dict(c) for s, c in have.items()},
        "by_domain": dict(collections.Counter(r["domain"] for r in rows)),
        "by_source": dict(collections.Counter(r["source"] for r in rows)),
        "by_company_reason": dict(collections.Counter(f"{r['domain']}:{r['company_reason'] or 'IMAGE_ONLY'}" for r in rows)),
        "groups": len({r["group"] for r in rows}),
        "near_duplicate_pairs": len(near),
        "review_queue_rows": len(review),
        "label_problems": 0, "leaking_groups": 0,
        "image_size": {"min_short_side": min(min(r["_q"]["width"], r["_q"]["height"]) for r in rows),
                       "max_long_side": max(max(r["_q"]["width"], r["_q"]["height"]) for r in rows)},
        "seed": SEED, "split_shares": {"train": 0.70, **SPLIT_SHARES},
    }
    json.dump(summary, open(os.path.join(out, "dataset_summary.json"), "w"), indent=1)
    return summary


if __name__ == "__main__":
    ap = argparse.ArgumentParser()
    here = os.path.dirname(os.path.abspath(__file__))
    ap.add_argument("--manifest", default=os.path.join(here, "manifest.jsonl"))
    ap.add_argument("--out", default="callverse_vision_dataset")
    ap.add_argument("--cache", default="vision_source_cache")
    a = ap.parse_args()
    s = build(a.manifest, a.out, a.cache)
    print(json.dumps(s, indent=1))

In [ ]:
%%writefile manifest.jsonl
{"id": "cv001", "cid": "c001", "domain": "shopnova", "source": "phones", "source_kind": "hf", "source_repo": "harshdadiya-wappnet/mobile_damage_diagnosis", "source_revision": "73bf263cf94711d5aa53feb871f701a9a16f93f7", "source_ref": {"path": "valid/Datacluster-Cracked-Screen-108-_jpg.rf.c6fef1a836540f88b982607851541011.jpg"}, "sha256_source": "3055a67143ad4f8391caf9d1aef54ad95619a1b5758d599b8b372f99a032edb1", "group": "phones:c001", "issue_type": "cracked_screen", "object": "smartphone", "confidence_level": "high", "description": "A white smartphone lying on fabric has a screen covered in long cracks spreading from the top corner.", "company_reason": "damaged_item", "customer_message": "My new phone arrived with the screen completely cracked, see the photo."}
{"id": "cv002", "cid": "c002", "domain": "shopnova", "source": "phones", "source_kind": "hf", "source_repo": "harshdadiya-wappnet/mobile_damage_diagnosis", "source_revision": "73bf263cf94711d5aa53feb871f701a9a16f93f7", "source_ref": {"path": "valid/Datacluster-Cracked-Screen-110-_jpg.rf.c0f34bf71a2e619c8f4d81e266f5bf3b.jpg"}, "sha256_source": "1d069973519d3669a635304f00debc42ad10b4c8d7cdc654efd9290d3d96325c", "group": "phones:c002", "issue_type": "unclear", "object": "smartphone", "confidence_level": "low", "description": "A white smartphone held in a hand; strong light glare hides most of the screen, so cracks cannot be confirmed.", "company_reason": "other", "customer_message": "Do you have a store in Tunis where I can come in person?"}
{"id": "cv003", "cid": "c003", "domain": "shopnova", "source": "phones", "source_kind": "hf", "source_repo": "harshdadiya-wappnet/mobile_damage_diagnosis", "source_revision": "73bf263cf94711d5aa53feb871f701a9a16f93f7", "source_ref": {"path": "valid/Datacluster-Cracked-Screen-118-_jpg.rf.329d46eb3645256e97f4f46f271070a4.jpg"}, "sha256_source": "29fe8b62b546893dc1517e5e1adc4f668e3acd835809cc01f687d81c899fe183", "group": "phones:c003", "issue_type": "cracked_screen", "object": "smartphone", "confidence_level": "high", "description": "A black smartphone in a case shows a shattered screen with many cracks, worst at the bottom corner.", "company_reason": "damaged_item", "customer_message": "The phone from order 4507 came broken, the glass is shattered."}
{"id": "cv004", "cid": "c004", "domain": "shopnova", "source": "phones", "source_kind": "hf", "source_repo": "harshdadiya-wappnet/mobile_damage_diagnosis", "source_revision": "73bf263cf94711d5aa53feb871f701a9a16f93f7", "source_ref": {"path": "valid/Datacluster-Cracked-Screen-132-_jpg.rf.34bcaa8cc941b94b3ec0e09e597614dd.jpg"}, "sha256_source": "46354086854e228a0db4ee602b115d4a03efee2a4dda795e25ec8353b0a3e880", "group": "phones:c004", "issue_type": "cracked_screen", "object": "smartphone", "confidence_level": "high", "description": "A silver smartphone held in a hand has several long cracks running across its dark screen.", "company_reason": "damaged_item", "customer_message": "I just unpacked my order and the screen is cracked all over."}
{"id": "cv005", "cid": "c005", "domain": "shopnova", "source": "phones", "source_kind": "hf", "source_repo": "harshdadiya-wappnet/mobile_damage_diagnosis", "source_revision": "73bf263cf94711d5aa53feb871f701a9a16f93f7", "source_ref": {"path": "valid/Datacluster-Cracked-Screen-138-_jpg.rf.c96fd3906c973cc02ef170c56b99bf35.jpg"}, "sha256_source": "3947ca75bdd71ba599e8f246b05fe01f41e0e728b8aed3b0b02f1e1a757f41f0", "group": "phones:c005", "issue_type": "cracked_screen", "object": "smartphone", "confidence_level": "high", "description": "A small smartphone on a dirty surface has a heavily cracked screen with broken glass near the bottom.", "company_reason": "damaged_item", "customer_message": "The phone I received is damaged, the screen is smashed at the bottom."}
{"id": "cv006", "cid": "c006", "domain": "shopnova", "source": "phones", "source_kind": "hf", "source_repo": "harshdadiya-wappnet/mobile_damage_diagnosis", "source_revision": "73bf263cf94711d5aa53feb871f701a9a16f93f7", "source_ref": {"path": "valid/Datacluster-Cracked-Screen-153-_jpg.rf.7188917f9712dbd635262996df3b4090.jpg"}, "sha256_source": "cce149d234e0cf1883b8ee1c058175f0da436d169f7c7c7878c303c8efb8af56", "group": "phones:c006", "issue_type": "cracked_screen", "object": "smartphone", "confidence_level": "medium", "description": "A black smartphone on a checked cloth shows thin crack lines on the upper part of the screen.", "company_reason": "late_delivery", "customer_message": "My order should have arrived days ago and it is still not here."}
{"id": "cv007", "cid": "c007", "domain": "shopnova", "source": "phones", "source_kind": "hf", "source_repo": "harshdadiya-wappnet/mobile_damage_diagnosis", "source_revision": "73bf263cf94711d5aa53feb871f701a9a16f93f7", "source_ref": {"path": "valid/Datacluster-Cracked-Screen-156-_jpg.rf.cfbb6f47ab976d02969bd86a9794368d.jpg"}, "sha256_source": "39b053ab3dc6ec88c20cad82ed8a1a4653cf2b38b1424a8419edbeadc949732f", "group": "phones:c007", "issue_type": "cracked_screen", "object": "smartphone", "confidence_level": "medium", "description": "A small white smartphone photographed from far away shows a cracked, spider-web pattern on its screen.", "company_reason": "other", "customer_message": "How do I change my account password?"}
{"id": "cv008", "cid": "c008", "domain": "shopnova", "source": "phones", "source_kind": "hf", "source_repo": "harshdadiya-wappnet/mobile_damage_diagnosis", "source_revision": "73bf263cf94711d5aa53feb871f701a9a16f93f7", "source_ref": {"path": "valid/Datacluster-Cracked-Screen-166-_jpg.rf.d1296c017d4215e321955cc4f4390800.jpg"}, "sha256_source": "12bf34e9d14975900f1c92488233db577e8833919120cb0604d6393a983c607e", "group": "phones:c008", "issue_type": "cracked_screen", "object": "smartphone", "confidence_level": "high", "description": "A black Microsoft phone shows long cracks across the screen, starting from the top right.", "company_reason": "damaged_item", "customer_message": "This is how the phone arrived today, the screen is broken."}
{"id": "cv009", "cid": "c009", "domain": "shopnova", "source": "phones", "source_kind": "hf", "source_repo": "harshdadiya-wappnet/mobile_damage_diagnosis", "source_revision": "73bf263cf94711d5aa53feb871f701a9a16f93f7", "source_ref": {"path": "valid/Datacluster-Cracked-Screen-169-_jpg.rf.120740da6f99217b8301a5d27b7dce64.jpg"}, "sha256_source": "b51bd9e648d13f2278b0b2030d248e6c00939e8dc97afbc93719ad0da882fbb0", "group": "phones:c009", "issue_type": "cracked_screen", "object": "smartphone", "confidence_level": "high", "description": "A smartphone on a patterned blanket has a completely shattered screen with dense cracks.", "company_reason": "damaged_item", "customer_message": "My order 4509 arrived damaged, look at the screen."}
{"id": "cv010", "cid": "c010", "domain": "shopnova", "source": "phones", "source_kind": "hf", "source_repo": "harshdadiya-wappnet/mobile_damage_diagnosis", "source_revision": "73bf263cf94711d5aa53feb871f701a9a16f93f7", "source_ref": {"path": "valid/Datacluster-Cracked-Screen-185-_jpg.rf.6c1bb4212f246a78a3f0369fa1b18eb5.jpg"}, "sha256_source": "34b1653c88ac50530433a847d419884e289a9008962eacbf485d146306d3fd64", "group": "phones:c010", "issue_type": "cracked_screen", "object": "smartphone", "confidence_level": "high", "description": "A black phone on a tiled floor has long cracks crossing the whole screen.", "company_reason": "refund_request", "customer_message": "I want my money back for order 4508."}
{"id": "cv011", "cid": "c011", "domain": "shopnova", "source": "phones", "source_kind": "hf", "source_repo": "harshdadiya-wappnet/mobile_damage_diagnosis", "source_revision": "73bf263cf94711d5aa53feb871f701a9a16f93f7", "source_ref": {"path": "valid/Datacluster-Cracked-Screen-187-_jpg.rf.7c2ca0595e25687330902cac14ea1b61.jpg"}, "sha256_source": "4221a2c6a8c26f28994bc8cffab38562e44278487614e7727f040bc42856f51c", "group": "phones:c011", "issue_type": "cracked_screen", "object": "smartphone", "confidence_level": "high", "description": "A black Samsung phone on blue fabric shows cracks across the screen and a broken lower edge.", "company_reason": "refund_request", "customer_message": "Please refund me for this phone, I paid with my card."}
{"id": "cv012", "cid": "c012", "domain": "shopnova", "source": "phones", "source_kind": "hf", "source_repo": "harshdadiya-wappnet/mobile_damage_diagnosis", "source_revision": "73bf263cf94711d5aa53feb871f701a9a16f93f7", "source_ref": {"path": "valid/Datacluster-Cracked-Screen-209-_jpg.rf.17ca23ce4985e04d5b5bc793aeb67eff.jpg"}, "sha256_source": "a4a542f222c2e7b944f6d95c279b65b5247dbe3158c0bf07aaec32d17e78c199", "group": "phones:c012", "issue_type": "cracked_screen", "object": "smartphone", "confidence_level": "medium", "description": "A smartphone held in a hand shows fine cracks and scratches over the dark screen.", "company_reason": "cancel_order", "customer_message": "Please cancel my other order 4513, I don't trust the delivery anymore."}
{"id": "cv013", "cid": "c013", "domain": "shopnova", "source": "phones", "source_kind": "hf", "source_repo": "harshdadiya-wappnet/mobile_damage_diagnosis", "source_revision": "73bf263cf94711d5aa53feb871f701a9a16f93f7", "source_ref": {"path": "valid/Datacluster-Cracked-Screen-217-_jpg.rf.af7f80ff21de25d8626b4a0ec7baeaef.jpg"}, "sha256_source": "1c639c4c07e78842142477c30dce50d7c863c1516ba77e4f99df26d20d95099f", "group": "phones:c013", "issue_type": "cracked_screen", "object": "smartphone", "confidence_level": "high", "description": "A black smartphone on a woven mat has many cracks covering the screen.", "company_reason": "refund_request", "customer_message": "Can I get a full refund for my order?"}
{"id": "cv014", "cid": "c014", "domain": "shopnova", "source": "phones", "source_kind": "hf", "source_repo": "harshdadiya-wappnet/mobile_damage_diagnosis", "source_revision": "73bf263cf94711d5aa53feb871f701a9a16f93f7", "source_ref": {"path": "valid/Datacluster-Cracked-Screen-220-_jpg.rf.db8f66e572a557dbb28bd7580d04864f.jpg"}, "sha256_source": "846c73619618db0c066d5a88ca6105a378ce0a1eac1c689b99ad4d819ebea1f5", "group": "phones:c014", "issue_type": "cracked_screen", "object": "smartphone", "confidence_level": "high", "description": "A black smartphone on fabric shows a screen covered with cracks from top to bottom.", "company_reason": "refund_request", "customer_message": "I don't want a replacement, just refund the money please."}
{"id": "cv015", "cid": "c015", "domain": "shopnova", "source": "phones", "source_kind": "hf", "source_repo": "harshdadiya-wappnet/mobile_damage_diagnosis", "source_revision": "73bf263cf94711d5aa53feb871f701a9a16f93f7", "source_ref": {"path": "valid/Datacluster-Cracked-Screen-249-_jpg.rf.bc68437141cfdc6bb7cd11aca324caff.jpg"}, "sha256_source": "dc93d9d7e2f6c2e6f4154da6a1216c09dc67926cc5581707db2465ebea7c2b64", "group": "phones:c015", "issue_type": "cracked_screen", "object": "smartphone", "confidence_level": "high", "description": "A white Samsung phone has cracks across the screen and a chipped corner at the bottom.", "company_reason": "return_item", "customer_message": "I'd like to send this phone back, how do I return it?"}
{"id": "cv016", "cid": "c016", "domain": "shopnova", "source": "phones", "source_kind": "hf", "source_repo": "harshdadiya-wappnet/mobile_damage_diagnosis", "source_revision": "73bf263cf94711d5aa53feb871f701a9a16f93f7", "source_ref": {"path": "valid/Datacluster-Cracked-Screen-250-_jpg.rf.932468e23c6c7da98c78b83ab096e6fa.jpg"}, "sha256_source": "80746d2d043ad11a2104af077d4d08b55c1c740c0d7c20b2f7760a5a27c976c9", "group": "phones:c016", "issue_type": "cracked_screen", "object": "smartphone", "confidence_level": "high", "description": "A black Panasonic phone shows a star-shaped crack on the left side of the screen.", "company_reason": "return_item", "customer_message": "How can I return the phone from order 4506?"}
{"id": "cv017", "cid": "c017", "domain": "shopnova", "source": "phones", "source_kind": "hf", "source_repo": "harshdadiya-wappnet/mobile_damage_diagnosis", "source_revision": "73bf263cf94711d5aa53feb871f701a9a16f93f7", "source_ref": {"path": "valid/Datacluster-Cracked-Screen-39-_jpg.rf.002862cfc784d950717f7fa77c6783c3.jpg"}, "sha256_source": "97dd4b138ee7cc191b06cd66e3443fdefc8928afa9b8219fdab523f7efa68948", "group": "phones:c017", "issue_type": "cracked_screen", "object": "smartphone", "confidence_level": "medium", "description": "A smartphone held in a hand shows thin cracks near the top of the screen; the photo is slightly dark.", "company_reason": "other", "customer_message": "Do you ship to Sousse?"}
{"id": "cv018", "cid": "c018", "domain": "shopnova", "source": "phones", "source_kind": "hf", "source_repo": "harshdadiya-wappnet/mobile_damage_diagnosis", "source_revision": "73bf263cf94711d5aa53feb871f701a9a16f93f7", "source_ref": {"path": "valid/0QabEUbf8eL8KM_1_HD_wallpaper_jpeg.rf.a0ba39e602bfce10937be56503b1fa92.jpg"}, "sha256_source": "72157c486f688c3b5dec46977cfb82ddf4fe494ae7e9225279c6362e874b5508", "group": "phones:c018", "issue_type": "screen_display_defect", "object": "smartphone", "confidence_level": "high", "description": "A smartphone that is switched on shows a bright vertical green line across the display.", "company_reason": "damaged_item", "customer_message": "The phone works but there is a green line on the screen since I unboxed it."}
{"id": "cv019", "cid": "c019", "domain": "shopnova", "source": "phones", "source_kind": "hf", "source_repo": "harshdadiya-wappnet/mobile_damage_diagnosis", "source_revision": "73bf263cf94711d5aa53feb871f701a9a16f93f7", "source_ref": {"path": "valid/0yIydu0_jNWBvM_1_HD_wallpaper_jpeg.rf.dd9d3090b00648d29cb312641c450e5d.jpg"}, "sha256_source": "6d0f41372717bf902fa397590f8a53ace2cced6fd57862e94b54dbc172c6ba54", "group": "phones:c019", "issue_type": "screen_display_defect", "object": "smartphone", "confidence_level": "high", "description": "A switched-on smartphone shows a thin vertical green line running down the left side of the display.", "company_reason": "refund_request", "customer_message": "Please give me my money back for order 4511."}
{"id": "cv020", "cid": "c020", "domain": "shopnova", "source": "phones", "source_kind": "hf", "source_repo": "harshdadiya-wappnet/mobile_damage_diagnosis", "source_revision": "73bf263cf94711d5aa53feb871f701a9a16f93f7", "source_ref": {"path": "valid/5fprbX5J7P8dIM_1_HD_wallpaper_jpeg.rf.1dd4d1f79276c6d2a1c02a411f0d1e6e.jpg"}, "sha256_source": "55731fd13b0b1044b8e8900ddd17be08773fac829273439ff08e90a4782ce32a", "group": "phones:c020", "issue_type": "screen_display_defect", "object": "smartphone", "confidence_level": "high", "description": "A smartphone held in a hand shows a bright vertical green line across the display.", "company_reason": "return_item", "customer_message": "I want to return this phone, what are the steps?"}
{"id": "cv021", "cid": "c021", "domain": "shopnova", "source": "phones", "source_kind": "hf", "source_repo": "harshdadiya-wappnet/mobile_damage_diagnosis", "source_revision": "73bf263cf94711d5aa53feb871f701a9a16f93f7", "source_ref": {"path": "valid/7MuMjwZgpeHCxM_1_HD_wallpaper_jpeg.rf.5e38f6eabd0c03f52251f6ac183d933c.jpg"}, "sha256_source": "0208bfe7cf0e0288dad74f227866bf7a1601907bf3f588ed7c317618cee9db68", "group": "phones:c021", "issue_type": "screen_display_defect", "object": "laptop", "confidence_level": "high", "description": "A laptop screen shows a thin vertical white line across a purple display.", "company_reason": "other", "customer_message": "Do you sell laptop chargers?"}
{"id": "cv022", "cid": "c022", "domain": "shopnova", "source": "phones", "source_kind": "hf", "source_repo": "harshdadiya-wappnet/mobile_damage_diagnosis", "source_revision": "73bf263cf94711d5aa53feb871f701a9a16f93f7", "source_ref": {"path": "valid/8v7dBgiydCxSZM_1_HD_wallpaper_jpeg.rf.fdf540b597cce14d6381ddebe503b328.jpg"}, "sha256_source": "875d64df56907a7f09199d848bdc3eb7c23397ed2dc41bc567fedab57e9ed767", "group": "phones:c022", "issue_type": "screen_display_defect", "object": "smartphone", "confidence_level": "high", "description": "A smartphone held in a hand shows two bright green vertical lines across the display.", "company_reason": "cancel_order", "customer_message": "I want to cancel order 4514 before it ships."}
{"id": "cv023", "cid": "c023", "domain": "shopnova", "source": "phones", "source_kind": "hf", "source_repo": "harshdadiya-wappnet/mobile_damage_diagnosis", "source_revision": "73bf263cf94711d5aa53feb871f701a9a16f93f7", "source_ref": {"path": "valid/001_004_27_jpg.rf.b3f463abab9671225fabd2ff4f54d050.jpg"}, "sha256_source": "03bed5c3119933165a670681b779c45874307f1dfd77325fc9554b4477d0f715", "group": "phone:001_004", "issue_type": "no_visible_issue", "object": "smartphone", "confidence_level": "medium", "description": "A foldable smartphone shows a set-up screen; no crack or display fault is visible in this photo.", "company_reason": "other", "customer_message": "Do you have a student discount?"}
{"id": "cv024", "cid": "c024", "domain": "shopnova", "source": "phones", "source_kind": "hf", "source_repo": "harshdadiya-wappnet/mobile_damage_diagnosis", "source_revision": "73bf263cf94711d5aa53feb871f701a9a16f93f7", "source_ref": {"path": "valid/005_016_02_jpg.rf.9616192b00a3377dad6d642685c2aa8a.jpg"}, "sha256_source": "6a533576c079ee721bf39449277b4595ea0cec14139520c689b075be6ad96907", "group": "phone:005_016", "issue_type": "no_visible_issue", "object": "smartphone", "confidence_level": "medium", "description": "A foldable smartphone on a wooden table shows a settings screen; no damage is visible in this photo.", "company_reason": "other", "customer_message": "How do I change the email on my account?"}
{"id": "cv025", "cid": "c025", "domain": "shopnova", "source": "phones", "source_kind": "hf", "source_repo": "harshdadiya-wappnet/mobile_damage_diagnosis", "source_revision": "73bf263cf94711d5aa53feb871f701a9a16f93f7", "source_ref": {"path": "valid/012_017_01_jpg.rf.e4393c23162f4c0775fe9ff0efdf8ac5.jpg"}, "sha256_source": "4b7af5e0ad3f8fb897d23b31ff051d67a43e8c1e7954a2dec9135a84c6aeb3a4", "group": "phone:012_017", "issue_type": "no_visible_issue", "object": "smartphone", "confidence_level": "medium", "description": "A smartphone on a white table shows a welcome screen; no damage is visible in this photo.", "company_reason": null, "customer_message": null}
{"id": "cv026", "cid": "c026", "domain": "shopnova", "source": "phones", "source_kind": "hf", "source_repo": "harshdadiya-wappnet/mobile_damage_diagnosis", "source_revision": "73bf263cf94711d5aa53feb871f701a9a16f93f7", "source_ref": {"path": "valid/016_016_jpg.rf.b0158442898bacc14664b1034849a865.jpg"}, "sha256_source": "6853e2ea8ad0c9e7f0ec863d2c27556270e48e98703525a0726e4c9c4ed7b274", "group": "phone:016_016", "issue_type": "no_visible_issue", "object": "smartphone", "confidence_level": "medium", "description": "A smartphone on a white table shows its home screen; no damage is visible in this photo.", "company_reason": null, "customer_message": null}
{"id": "cv027", "cid": "c027", "domain": "shopnova", "source": "phones", "source_kind": "hf", "source_repo": "harshdadiya-wappnet/mobile_damage_diagnosis", "source_revision": "73bf263cf94711d5aa53feb871f701a9a16f93f7", "source_ref": {"path": "valid/020_006_24_jpg.rf.618d6015273358f3ca6b9daf6e3c63f2.jpg"}, "sha256_source": "a6cd1b0e3d80c76b339ce1bd9f0b47f26af113f5322f27cacbc5aa142017c2de", "group": "phone:020_006", "issue_type": "no_visible_issue", "object": "smartphone", "confidence_level": "medium", "description": "A foldable smartphone shows a set-up screen; no crack or display fault is visible in this photo.", "company_reason": null, "customer_message": null}
{"id": "cv028", "cid": "c028", "domain": "shopnova", "source": "abo", "source_kind": "hf_parquet", "source_repo": "suvadityamuk/amazon-berkeley-objects", "source_revision": "b3eef2456b77d99ff7d6a9a3be758168f2e4ffa8", "source_ref": {"parquet": "images_small/train-00002.parquet", "key_column": "image_id", "key": "15c88dc8"}, "sha256_source": "cc18ec28912b623d68747db04608ad3ccdd288ac5de9f11a7e9b3523fd15f4d8", "group": "abo:c028", "issue_type": "no_visible_issue", "object": "suitcase", "confidence_level": "high", "description": "An open hard-shell suitcase shown empty with its inner straps; no damage is visible.", "company_reason": "late_delivery", "customer_message": "The suitcase I ordered was promised last week and still has not come."}
{"id": "cv029", "cid": "c029", "domain": "shopnova", "source": "abo", "source_kind": "hf_parquet", "source_repo": "suvadityamuk/amazon-berkeley-objects", "source_revision": "b3eef2456b77d99ff7d6a9a3be758168f2e4ffa8", "source_ref": {"parquet": "images_small/train-00002.parquet", "key_column": "image_id", "key": "3e8bb6d4"}, "sha256_source": "f46e6cad72fbaec51d47d1524dc9a963590d9b6ca1925645a08be8522b8f120c", "group": "abo:c029", "issue_type": "no_visible_issue", "object": "shoe", "confidence_level": "high", "description": "A close view of a navy and grey sports shoe; no damage is visible.", "company_reason": "return_item", "customer_message": "I ordered headphones but received these shoes, I want to send them back."}
{"id": "cv030", "cid": "c030", "domain": "shopnova", "source": "abo", "source_kind": "hf_parquet", "source_repo": "suvadityamuk/amazon-berkeley-objects", "source_revision": "b3eef2456b77d99ff7d6a9a3be758168f2e4ffa8", "source_ref": {"parquet": "images_small/train-00002.parquet", "key_column": "image_id", "key": "3e9af99b"}, "sha256_source": "d8d35086336dd1ebea099c95b6d1b5c345d06634f610b3cf2d6d2d9ec0616270", "group": "abo:c030", "issue_type": "no_visible_issue", "object": "shoe", "confidence_level": "high", "description": "A black knitted sports shoe with a white sole; no damage is visible.", "company_reason": "return_item", "customer_message": "These shoes don't fit me, can I return them?"}
{"id": "cv031", "cid": "c031", "domain": "shopnova", "source": "abo", "source_kind": "hf_parquet", "source_repo": "suvadityamuk/amazon-berkeley-objects", "source_revision": "b3eef2456b77d99ff7d6a9a3be758168f2e4ffa8", "source_ref": {"parquet": "images_small/train-00002.parquet", "key_column": "image_id", "key": "4a6ce5a8"}, "sha256_source": "025ef2b3dbed91403b79dba907634bd64dfae7e518a7fdf41022375b50a1db09", "group": "abo:c031", "issue_type": "no_visible_issue", "object": "armchair", "confidence_level": "high", "description": "A blue and white checked armchair; no damage is visible.", "company_reason": "late_delivery", "customer_message": "The armchair is very late, the delivery date passed five days ago."}
{"id": "cv032", "cid": "c032", "domain": "shopnova", "source": "abo", "source_kind": "hf_parquet", "source_repo": "suvadityamuk/amazon-berkeley-objects", "source_revision": "b3eef2456b77d99ff7d6a9a3be758168f2e4ffa8", "source_ref": {"parquet": "images_small/train-00002.parquet", "key_column": "image_id", "key": "4a92c46f"}, "sha256_source": "9ee592d3a164aa3db06d852992a6989f6c3a594f85899efe4b98e1b0910e72d5", "group": "abo:c032", "issue_type": "no_visible_issue", "object": "backpack", "confidence_level": "high", "description": "A tan backpack with a small matching pouch; no damage is visible.", "company_reason": "return_item", "customer_message": "This is not the bag I ordered, I want to return it."}
{"id": "cv033", "cid": "c033", "domain": "shopnova", "source": "abo", "source_kind": "hf_parquet", "source_repo": "suvadityamuk/amazon-berkeley-objects", "source_revision": "b3eef2456b77d99ff7d6a9a3be758168f2e4ffa8", "source_ref": {"parquet": "images_small/train-00002.parquet", "key_column": "image_id", "key": "4a9af532"}, "sha256_source": "40bcdb554099a1adee316bb9d9d33802f79c7dee5523a0e74ddddc82119f4afb", "group": "abo:c033", "issue_type": "no_visible_issue", "object": "pliers", "confidence_level": "high", "description": "A pair of adjustable pliers with black handles; no damage is visible.", "company_reason": "late_delivery", "customer_message": "My order 4512 was due on October 3rd and I am still waiting."}
{"id": "cv034", "cid": "c034", "domain": "shopnova", "source": "abo", "source_kind": "hf_parquet", "source_repo": "suvadityamuk/amazon-berkeley-objects", "source_revision": "b3eef2456b77d99ff7d6a9a3be758168f2e4ffa8", "source_ref": {"parquet": "images_small/train-00002.parquet", "key_column": "image_id", "key": "bd6ee18e"}, "sha256_source": "fe191968ec5262269b62c7897f21e2ed910fd0322b8dd040cb89ec2ffbdfed45", "group": "abo:c034", "issue_type": "no_visible_issue", "object": "smartphone", "confidence_level": "high", "description": "A smartphone in a dark patterned case, shown from the front and back; no damage is visible.", "company_reason": "late_delivery", "customer_message": "The delivery is late, it has been a week past the date you gave me."}
{"id": "cv035", "cid": "c036", "domain": "shopnova", "source": "abo", "source_kind": "hf_parquet", "source_repo": "suvadityamuk/amazon-berkeley-objects", "source_revision": "b3eef2456b77d99ff7d6a9a3be758168f2e4ffa8", "source_ref": {"parquet": "images_small/train-00002.parquet", "key_column": "image_id", "key": "f26034b6"}, "sha256_source": "38979b65dd85d574f0c1b9def8aa96ebc9bdc4d0d0a2ce46ca17e2d247cce0a9", "group": "abo:c036", "issue_type": "no_visible_issue", "object": "shoe", "confidence_level": "high", "description": "A pair of black low sneakers with white soles; no damage is visible.", "company_reason": "return_item", "customer_message": "I received the wrong colour, can I send these sneakers back?"}
{"id": "cv036", "cid": "c037", "domain": "shopnova", "source": "abo", "source_kind": "hf_parquet", "source_repo": "suvadityamuk/amazon-berkeley-objects", "source_revision": "b3eef2456b77d99ff7d6a9a3be758168f2e4ffa8", "source_ref": {"parquet": "images_small/train-00002.parquet", "key_column": "image_id", "key": "f2b1c185"}, "sha256_source": "c8e58da7c03ced04bccccae7ca16b66339f7dd5dffd4df088eb899be797bfc34", "group": "abo:c037", "issue_type": "no_visible_issue", "object": "wallet", "confidence_level": "high", "description": "A closed pink wallet with a red clasp; no damage is visible.", "company_reason": "cancel_order", "customer_message": "Can you cancel the wallet I ordered yesterday?"}
{"id": "cv037", "cid": "c038", "domain": "shopnova", "source": "abo", "source_kind": "hf_parquet", "source_repo": "suvadityamuk/amazon-berkeley-objects", "source_revision": "b3eef2456b77d99ff7d6a9a3be758168f2e4ffa8", "source_ref": {"parquet": "images_small/train-00002.parquet", "key_column": "image_id", "key": "f2b8715e"}, "sha256_source": "65735d18a116498dc7439343d6047640919434c9eecfa080d77bcf0408a91659", "group": "abo:c038", "issue_type": "no_visible_issue", "object": "paper_shredder", "confidence_level": "high", "description": "A black paper shredder with its instruction papers; no damage is visible.", "company_reason": "cancel_order", "customer_message": "I ordered this shredder by mistake, please cancel it."}
{"id": "cv038", "cid": "c039", "domain": "shopnova", "source": "abo", "source_kind": "hf_parquet", "source_repo": "suvadityamuk/amazon-berkeley-objects", "source_revision": "b3eef2456b77d99ff7d6a9a3be758168f2e4ffa8", "source_ref": {"parquet": "images_small/train-00002.parquet", "key_column": "image_id", "key": "f817952c"}, "sha256_source": "83059be9294b46e0692e2761c14df8d9fd343ba47621c9f53d42705a20c9646d", "group": "abo:c039", "issue_type": "no_visible_issue", "object": "cable", "confidence_level": "high", "description": "A black DVI to HDMI cable with both connectors shown; no damage is visible.", "company_reason": "order_status", "customer_message": "Where is my order? I ordered this cable last week."}
{"id": "cv039", "cid": "c040", "domain": "shopnova", "source": "abo", "source_kind": "hf_parquet", "source_repo": "suvadityamuk/amazon-berkeley-objects", "source_revision": "b3eef2456b77d99ff7d6a9a3be758168f2e4ffa8", "source_ref": {"parquet": "images_small/train-00002.parquet", "key_column": "image_id", "key": "f8346417"}, "sha256_source": "f499c827c97333d375f40c54a999c19cb6e9fa1f47d664048b5dacf9301f4f94", "group": "abo:c040", "issue_type": "no_visible_issue", "object": "sofa", "confidence_level": "high", "description": "A cream two-seat sofa; no damage is visible.", "company_reason": "order_status", "customer_message": "What is the status of order 4505?"}
{"id": "cv040", "cid": "c041", "domain": "shopnova", "source": "abo", "source_kind": "hf_parquet", "source_repo": "suvadityamuk/amazon-berkeley-objects", "source_revision": "b3eef2456b77d99ff7d6a9a3be758168f2e4ffa8", "source_ref": {"parquet": "images_small/train-00002.parquet", "key_column": "image_id", "key": "f8407059"}, "sha256_source": "bf18ec954b5e5f49b9cd6b8e71a0796bcf22fba1ace978017bf6149101f4af11", "group": "abo:c041", "issue_type": "no_visible_issue", "object": "bed_sheets", "confidence_level": "high", "description": "A folded set of purple bed sheets; no damage is visible.", "company_reason": "order_status", "customer_message": "Has my package been shipped yet?"}
{"id": "cv041", "cid": "c042", "domain": "shopnova", "source": "abo", "source_kind": "hf_parquet", "source_repo": "suvadityamuk/amazon-berkeley-objects", "source_revision": "b3eef2456b77d99ff7d6a9a3be758168f2e4ffa8", "source_ref": {"parquet": "images_small/train-00002.parquet", "key_column": "image_id", "key": "f8720c04"}, "sha256_source": "e655838bb00c36b66e49b87e4f7f09dca5277146d3f7f8088ed0f8da64916e75", "group": "abo:c042", "issue_type": "no_visible_issue", "object": "sandal", "confidence_level": "high", "description": "A gold and brown sandal; no damage is visible.", "company_reason": "return_item", "customer_message": "The sandals are the wrong size, how do I return them?"}
{"id": "cv042", "cid": "c043", "domain": "shopnova", "source": "abo", "source_kind": "hf_parquet", "source_repo": "suvadityamuk/amazon-berkeley-objects", "source_revision": "b3eef2456b77d99ff7d6a9a3be758168f2e4ffa8", "source_ref": {"parquet": "images_small/train-00002.parquet", "key_column": "image_id", "key": "4a0cd540"}, "sha256_source": "72b84f06e357e41241ed2b79c7d32fb646b06553a6899884eac8ed8178bdd209", "group": "abo:c043", "issue_type": "no_visible_issue", "object": "filament_spool", "confidence_level": "high", "description": "A spool of pink 3D-printer filament; no damage is visible.", "company_reason": "order_status", "customer_message": "Can you tell me where my order is right now?"}
{"id": "cv043", "cid": "c044", "domain": "shopnova", "source": "cardboard", "source_kind": "hf", "source_repo": "Gabriel8/cardboard-box-anomaly-detection", "source_revision": "a1c01d16c4131ba049d3adace44b870495c25dd3", "source_ref": {"path": "test/good/cam1_box09_pos03_ver00_loc-chao.jpg"}, "sha256_source": "1a8462db226f9ca6e608101cb88f26ad57ea6ad217bca11466b19cf78948304a", "group": "box:box09", "issue_type": "no_visible_issue", "object": "cardboard_box", "confidence_level": "high", "description": "A flattened cardboard box lying on a concrete floor; no tears or crushed parts are visible.", "company_reason": "cancel_order", "customer_message": "Cancel my order please, I found it cheaper elsewhere."}
{"id": "cv044", "cid": "c045", "domain": "shopnova", "source": "cardboard", "source_kind": "hf", "source_repo": "Gabriel8/cardboard-box-anomaly-detection", "source_revision": "a1c01d16c4131ba049d3adace44b870495c25dd3", "source_ref": {"path": "test/good/cam1_box15_pos03_ver01_loc-chao.jpg"}, "sha256_source": "68aad253dcf8426048c5fe60917822af41be83890dd3e696cde04e2a27a2df08", "group": "box:box15", "issue_type": "no_visible_issue", "object": "cardboard_box", "confidence_level": "high", "description": "A flattened cardboard box on a concrete floor; the edges and flaps look intact.", "company_reason": "cancel_order", "customer_message": "How do I cancel order 4515?"}
{"id": "cv045", "cid": "c046", "domain": "shopnova", "source": "cardboard", "source_kind": "hf", "source_repo": "Gabriel8/cardboard-box-anomaly-detection", "source_revision": "a1c01d16c4131ba049d3adace44b870495c25dd3", "source_ref": {"path": "test/good/cam1_box19_pos02_ver00_loc-chao.jpg"}, "sha256_source": "8c9f4ed1748f204577f5ccecd1f9f6845af3ba57432440a31359d13ca9151be2", "group": "box:box19", "issue_type": "no_visible_issue", "object": "cardboard_box", "confidence_level": "high", "description": "A flattened cardboard box on a concrete floor; no tears are visible.", "company_reason": "order_status", "customer_message": "Is this box my order? The tracking says delivered."}
{"id": "cv046", "cid": "c047", "domain": "shopnova", "source": "cardboard", "source_kind": "hf", "source_repo": "Gabriel8/cardboard-box-anomaly-detection", "source_revision": "a1c01d16c4131ba049d3adace44b870495c25dd3", "source_ref": {"path": "test/good/cam2_box09_pos03_ver00_loc-chao.jpg"}, "sha256_source": "af25ba48ded9658c8ac199e5c150ef8bd8a1b25f55fe5f403df8863b7846e095", "group": "box:box09", "issue_type": "no_visible_issue", "object": "cardboard_box", "confidence_level": "high", "description": "A partly folded cardboard box on a concrete floor; no tears or holes are visible.", "company_reason": "order_status", "customer_message": "When will my order 4512 be sent?"}
{"id": "cv047", "cid": "c048", "domain": "shopnova", "source": "cardboard", "source_kind": "hf", "source_repo": "Gabriel8/cardboard-box-anomaly-detection", "source_revision": "a1c01d16c4131ba049d3adace44b870495c25dd3", "source_ref": {"path": "test/good/cam2_box17_pos01_ver00_loc-chao.jpg"}, "sha256_source": "d9efefcb375b40f12364fc553cab34c3509a2f4143bbd0f09a873340687426f1", "group": "box:box17", "issue_type": "no_visible_issue", "object": "cardboard_box", "confidence_level": "high", "description": "A long flattened cardboard box on a concrete floor; the edges look intact.", "company_reason": "order_status", "customer_message": "Can you check the status of my parcel?"}
{"id": "cv048", "cid": "c049", "domain": "shopnova", "source": "cardboard", "source_kind": "hf", "source_repo": "Gabriel8/cardboard-box-anomaly-detection", "source_revision": "a1c01d16c4131ba049d3adace44b870495c25dd3", "source_ref": {"path": "test/good/cam2_box17_pos03_ver00_loc-chao.jpg"}, "sha256_source": "e99803a8134ef409600b9669dc0fa387001259c6df296d9ee9f85140e39e01b5", "group": "box:box17", "issue_type": "no_visible_issue", "object": "cardboard_box", "confidence_level": "high", "description": "A flattened cardboard box on a concrete floor; no damage is visible.", "company_reason": "order_status", "customer_message": "Where is my package? I have not seen any update."}
{"id": "cv049", "cid": "c050", "domain": "shopnova", "source": "cardboard", "source_kind": "hf", "source_repo": "Gabriel8/cardboard-box-anomaly-detection", "source_revision": "a1c01d16c4131ba049d3adace44b870495c25dd3", "source_ref": {"path": "test/bad/cam1_box34_pos02_ver00_loc-b1.jpg"}, "sha256_source": "7bc706bacba9c8e43fa2d637d76aeb0c93c9f3956a2e6c84b3e9491d1acbf774", "group": "box:box34", "issue_type": "damaged_packaging", "object": "cardboard_box", "confidence_level": "high", "description": "A flattened cardboard box on a roller conveyor is torn, with a large ripped section on the right.", "company_reason": "damaged_item", "customer_message": "The box arrived torn open and the item inside is broken."}
{"id": "cv050", "cid": "c051", "domain": "shopnova", "source": "cardboard", "source_kind": "hf", "source_repo": "Gabriel8/cardboard-box-anomaly-detection", "source_revision": "a1c01d16c4131ba049d3adace44b870495c25dd3", "source_ref": {"path": "test/bad/cam1_box35_pos02_ver00_loc-b1.jpg"}, "sha256_source": "e7df5d98b658112c7760aa854584aed61b43756d88a2baf19a9d23d506c0bf90", "group": "box:box35", "issue_type": "damaged_packaging", "object": "cardboard_box", "confidence_level": "high", "description": "A flattened cardboard box on a roller conveyor has a torn and folded-back corner.", "company_reason": "damaged_item", "customer_message": "My parcel came ripped and crushed, the product inside is damaged."}
{"id": "cv051", "cid": "c052", "domain": "shopnova", "source": "cardboard", "source_kind": "hf", "source_repo": "Gabriel8/cardboard-box-anomaly-detection", "source_revision": "a1c01d16c4131ba049d3adace44b870495c25dd3", "source_ref": {"path": "test/bad/cam1_box36_pos02_ver00_loc-b1.jpg"}, "sha256_source": "e8ab587cfab272343228d5ac57dd65978bf02522ddd0f60a8b5d9b06165e4816", "group": "box:box36", "issue_type": "damaged_packaging", "object": "cardboard_box", "confidence_level": "high", "description": "A flattened cardboard box on a roller conveyor is ripped near the middle and on one edge.", "company_reason": "refund_request", "customer_message": "Refund my order please, I am not happy with it."}
{"id": "cv052", "cid": "c053", "domain": "shopnova", "source": "cardboard", "source_kind": "hf", "source_repo": "Gabriel8/cardboard-box-anomaly-detection", "source_revision": "a1c01d16c4131ba049d3adace44b870495c25dd3", "source_ref": {"path": "test/bad/cam1_box37_pos02_ver00_loc-b1.jpg"}, "sha256_source": "001a8d9406f90eaa03d6f5312ccabd98f86066aabaf332ec123e5dedb7f80da4", "group": "box:box37", "issue_type": "damaged_packaging", "object": "cardboard_box", "confidence_level": "high", "description": "A flattened cardboard box on a roller conveyor has a missing, torn-off piece at the top.", "company_reason": "refund_request", "customer_message": "I would like a refund for order 4510."}
{"id": "cv053", "cid": "c054", "domain": "shopnova", "source": "cardboard", "source_kind": "hf", "source_repo": "Gabriel8/cardboard-box-anomaly-detection", "source_revision": "a1c01d16c4131ba049d3adace44b870495c25dd3", "source_ref": {"path": "test/bad/cam1_box38_pos02_ver00_loc-chao.jpg"}, "sha256_source": "ee9c014af30ce0aaaaeb2cd3b06ea18aa4ea5912d9203be47ac17fd090f8a12b", "group": "box:box38", "issue_type": "damaged_packaging", "object": "cardboard_box", "confidence_level": "high", "description": "A flattened cardboard box on the floor has a torn edge on the left side.", "company_reason": "cancel_order", "customer_message": "I need to cancel my order, it is not needed anymore."}
{"id": "cv054", "cid": "c055", "domain": "shopnova", "source": "cardboard", "source_kind": "hf", "source_repo": "Gabriel8/cardboard-box-anomaly-detection", "source_revision": "a1c01d16c4131ba049d3adace44b870495c25dd3", "source_ref": {"path": "test/bad/cam2_box34_pos03_ver00_loc-chao.jpg"}, "sha256_source": "48c1f0738aa00d3e51fc3228ff6f3a7e9f35e6e36fb98d43743a8cfc167d4f68", "group": "box:box34", "issue_type": "damaged_packaging", "object": "cardboard_box", "confidence_level": "medium", "description": "A flattened cardboard box on the floor shows several tears and cut lines across its panels.", "company_reason": "late_delivery", "customer_message": "My parcel is five days late, why is it taking so long?"}
{"id": "cv055", "cid": "c056", "domain": "shopnova", "source": "cardboard", "source_kind": "hf", "source_repo": "Gabriel8/cardboard-box-anomaly-detection", "source_revision": "a1c01d16c4131ba049d3adace44b870495c25dd3", "source_ref": {"path": "test/bad/cam2_box37_pos01_ver00_loc-b1.jpg"}, "sha256_source": "5ceb90eda6d87a3102c63a8e61c8942ee22b0159507f614f4d73795b2d7364c0", "group": "box:box37", "issue_type": "damaged_packaging", "object": "cardboard_box", "confidence_level": "high", "description": "A cardboard box on a roller conveyor is badly torn, with pieces ripped off its edges.", "company_reason": "damaged_item", "customer_message": "Look at the packaging, it arrived torn apart and the item is broken."}
{"id": "cv056", "cid": "c057", "domain": "shopnova", "source": "cardboard", "source_kind": "hf", "source_repo": "Gabriel8/cardboard-box-anomaly-detection", "source_revision": "a1c01d16c4131ba049d3adace44b870495c25dd3", "source_ref": {"path": "test/bad/cam2_box38_pos01_ver00_loc-chao.jpg"}, "sha256_source": "3a12ba3c680b3a1dfcb6d6eae932780bf5966924755201eb8b3c0ecb006a0dd3", "group": "box:box38", "issue_type": "damaged_packaging", "object": "cardboard_box", "confidence_level": "medium", "description": "A flattened cardboard box on the floor has a torn lower edge.", "company_reason": "late_delivery", "customer_message": "The delivery was supposed to come on Monday and nothing arrived."}
{"id": "cv057", "cid": "c058", "domain": "atlas", "source": "dlsd", "source_kind": "github", "source_repo": "github6-dev/luggage-damage-datasets", "source_revision": "ce02f2115b6796592340e475c1650a0d3a8d56a2", "source_ref": {"path": "DLSD/images/val/CANCZ102106-1.jpg"}, "sha256_source": "6f2f0ee470791b277028baafdd9e7d882017f44500aac4872a6fe2c91efa88b6", "group": "dlsd:CANCZ102106", "issue_type": "damaged_luggage", "object": "suitcase", "confidence_level": "high", "description": "A silver hard-shell suitcase has a large hole broken into its shell.", "company_reason": "compensation_request", "customer_message": "My suitcase arrived broken after the flight, I want compensation."}
{"id": "cv058", "cid": "c059", "domain": "atlas", "source": "dlsd", "source_kind": "github", "source_repo": "github6-dev/luggage-damage-datasets", "source_revision": "ce02f2115b6796592340e475c1650a0d3a8d56a2", "source_ref": {"path": "DLSD/images/val/CANCZ102606-1.jpg"}, "sha256_source": "7a559b837bf15e54f74409322bc3420f37c7bc59cd1b0b0f8002bd7a4150affc", "group": "dlsd:CANCZ102606", "issue_type": "damaged_luggage", "object": "suitcase", "confidence_level": "high", "description": "A silver hard-shell suitcase is cracked open at the corner, showing the contents inside.", "company_reason": "compensation_request", "customer_message": "The airline destroyed my luggage, how do I claim compensation?"}
{"id": "cv059", "cid": "c060", "domain": "atlas", "source": "dlsd", "source_kind": "github", "source_repo": "github6-dev/luggage-damage-datasets", "source_revision": "ce02f2115b6796592340e475c1650a0d3a8d56a2", "source_ref": {"path": "DLSD/images/val/CANCZ30330-1.jpg"}, "sha256_source": "a9b0164384449bc6790e4256fafa719edb837d96f8114c98b253b9ac3f1cf28d", "group": "dlsd:CANCZ30330", "issue_type": "damaged_luggage", "object": "suitcase", "confidence_level": "high", "description": "A blue suitcase has a large broken hole in the corner of its shell.", "company_reason": "compensation_request", "customer_message": "Under passenger rights I should get money for my damaged bag."}
{"id": "cv060", "cid": "c061", "domain": "atlas", "source": "dlsd", "source_kind": "github", "source_repo": "github6-dev/luggage-damage-datasets", "source_revision": "ce02f2115b6796592340e475c1650a0d3a8d56a2", "source_ref": {"path": "DLSD/images/val/CANCZ30119-1.jpg"}, "sha256_source": "e3238a6077ecaa6458f5a8a90f742ea6816c1c72892e87aec180f24f40aa7e64", "group": "dlsd:CANCZ30119", "issue_type": "damaged_luggage", "object": "suitcase", "confidence_level": "high", "description": "A grey suitcase with airline tags has a broken hole near the lower corner.", "company_reason": "compensation_request", "customer_message": "I'm entitled to compensation for my broken suitcase, how do I claim it?"}
{"id": "cv061", "cid": "c062", "domain": "atlas", "source": "dlsd", "source_kind": "github", "source_repo": "github6-dev/luggage-damage-datasets", "source_revision": "ce02f2115b6796592340e475c1650a0d3a8d56a2", "source_ref": {"path": "DLSD/images/val/CANCZ103300-1.jpg"}, "sha256_source": "402c3207a60fd32b34f3339acef19aa2bded6eba3138bcfa3996b888ad293b3b", "group": "dlsd:CANCZ103300", "issue_type": "damaged_luggage", "object": "suitcase", "confidence_level": "medium", "description": "A pink suitcase wrapped in plastic has a torn section where the inner lining shows.", "company_reason": "compensation_request", "customer_message": "My bag was torn during the flight, I want to be compensated."}
{"id": "cv062", "cid": "c063", "domain": "atlas", "source": "dlsd", "source_kind": "github", "source_repo": "github6-dev/luggage-damage-datasets", "source_revision": "ce02f2115b6796592340e475c1650a0d3a8d56a2", "source_ref": {"path": "DLSD/images/val/CANCZ30321-1.jpg"}, "sha256_source": "086c51899e371ba006e5b463be4f0037f6dface2efe026b963907615976eb241", "group": "dlsd:CANCZ30321", "issue_type": "damaged_luggage", "object": "suitcase", "confidence_level": "high", "description": "A purple hard-shell suitcase has a round hole punched into its side.", "company_reason": "compensation_request", "customer_message": "I want compensation for the hole in my suitcase."}
{"id": "cv063", "cid": "c064", "domain": "atlas", "source": "dlsd", "source_kind": "github", "source_repo": "github6-dev/luggage-damage-datasets", "source_revision": "ce02f2115b6796592340e475c1650a0d3a8d56a2", "source_ref": {"path": "DLSD/images/val/CANCZ101699-1.jpg"}, "sha256_source": "6b819c9815f890d132262c7a810595ff69014539c99c0b32c1f3d4a3e08f0d10", "group": "dlsd:CANCZ101699", "issue_type": "damaged_luggage", "object": "suitcase", "confidence_level": "high", "description": "A pink suitcase with an airline tag has a large hole in its shell with the lining showing.", "company_reason": "compensation_request", "customer_message": "How do I claim compensation? My luggage came back damaged."}
{"id": "cv064", "cid": "c065", "domain": "atlas", "source": "dlsd", "source_kind": "github", "source_repo": "github6-dev/luggage-damage-datasets", "source_revision": "ce02f2115b6796592340e475c1650a0d3a8d56a2", "source_ref": {"path": "DLSD/images/val/CANCZ29751-1.jpg"}, "sha256_source": "7a671b66eb6eb9c8240e0427a0bd7f856a7b7f3bd6aa5dd9e02559a1d348fe0c", "group": "dlsd:CANCZ29751", "issue_type": "damaged_luggage", "object": "suitcase", "confidence_level": "high", "description": "A black hard-shell suitcase has a large broken section where the lining shows through.", "company_reason": "compensation_request", "customer_message": "My suitcase was smashed on flight 7011, I want compensation."}
{"id": "cv065", "cid": "c066", "domain": "atlas", "source": "dlsd", "source_kind": "github", "source_repo": "github6-dev/luggage-damage-datasets", "source_revision": "ce02f2115b6796592340e475c1650a0d3a8d56a2", "source_ref": {"path": "DLSD/images/val/CANCZ29824-1.jpg"}, "sha256_source": "68f40cdf10a3b8bf142f99f8ace1859e8ec62c93e257b70ff45f4eed6b78f3f6", "group": "dlsd:CANCZ29824", "issue_type": "damaged_luggage", "object": "suitcase", "confidence_level": "medium", "description": "A dark green suitcase with airline tags has a dented and cracked corner.", "company_reason": "compensation_request", "customer_message": "I demand compensation for my damaged luggage."}
{"id": "cv066", "cid": "c067", "domain": "atlas", "source": "dlsd", "source_kind": "github", "source_repo": "github6-dev/luggage-damage-datasets", "source_revision": "ce02f2115b6796592340e475c1650a0d3a8d56a2", "source_ref": {"path": "DLSD/images/val/CANCZ104404-1.jpg"}, "sha256_source": "0487d7b0add1ab3b12e44b406bb597cd072f224281b269e3094526f053731bf8", "group": "dlsd:CANCZ104404", "issue_type": "damaged_luggage", "object": "suitcase", "confidence_level": "high", "description": "A suitcase has a broken hole at the top corner of its shell.", "company_reason": "compensation_request", "customer_message": "What compensation do I get for a broken suitcase?"}
{"id": "cv067", "cid": "c068", "domain": "atlas", "source": "dlsd", "source_kind": "github", "source_repo": "github6-dev/luggage-damage-datasets", "source_revision": "ce02f2115b6796592340e475c1650a0d3a8d56a2", "source_ref": {"path": "DLSD/images/val/CANCZ106241-1.jpg"}, "sha256_source": "2b278e75de68399a50a81b23a2d989e904cc68db6cbb8d2560be3790acaf07c2", "group": "dlsd:CANCZ106241", "issue_type": "damaged_luggage", "object": "suitcase", "confidence_level": "high", "description": "A black ribbed suitcase has a hole torn into the shell near the wheel.", "company_reason": "special_assistance", "customer_message": "I use a wheelchair, can I get help at the airport with my bag?"}
{"id": "cv068", "cid": "c069", "domain": "atlas", "source": "dlsd", "source_kind": "github", "source_repo": "github6-dev/luggage-damage-datasets", "source_revision": "ce02f2115b6796592340e475c1650a0d3a8d56a2", "source_ref": {"path": "DLSD/images/val/CANCZ102460-1.jpg"}, "sha256_source": "07b67684c8ec96793488ffb6bde39aed915c0a0c10632a599e7d6c79732a157b", "group": "dlsd:CANCZ102460", "issue_type": "damaged_luggage", "object": "suitcase", "confidence_level": "high", "description": "A purple suitcase has a cracked and torn shell next to its airline tag.", "company_reason": "special_assistance", "customer_message": "My mother needs assistance boarding the plane."}
{"id": "cv069", "cid": "c070", "domain": "atlas", "source": "dlsd", "source_kind": "github", "source_repo": "github6-dev/luggage-damage-datasets", "source_revision": "ce02f2115b6796592340e475c1650a0d3a8d56a2", "source_ref": {"path": "DLSD/images/val/CANCZ101801-1.jpg"}, "sha256_source": "649c65ecce5fd01d627d2af24f3074bf5772722df1c88f50692aac49e363267d", "group": "dlsd:CANCZ101801", "issue_type": "damaged_luggage", "object": "suitcase", "confidence_level": "high", "description": "A silver ribbed suitcase has a split in its shell near the airline tag.", "company_reason": "other", "customer_message": "Do you sell travel insurance?"}
{"id": "cv070", "cid": "c071", "domain": "atlas", "source": "dlsd", "source_kind": "github", "source_repo": "github6-dev/luggage-damage-datasets", "source_revision": "ce02f2115b6796592340e475c1650a0d3a8d56a2", "source_ref": {"path": "DLSD/images/val/CANCZ103345-1.jpg"}, "sha256_source": "ce07b9d0f2dbb373361ab6a08b8116163dbb649d1d1f787b1b1ea26303ce3310", "group": "dlsd:CANCZ103345", "issue_type": "unclear", "object": "suitcase", "confidence_level": "low", "description": "A silver ribbed suitcase is mostly hidden by a hand holding boarding papers; damage cannot be confirmed.", "company_reason": "other", "customer_message": "How do I change the email on my account?"}
{"id": "cv071", "cid": "c072", "domain": "atlas", "source": "dlsd", "source_kind": "github", "source_repo": "github6-dev/luggage-damage-datasets", "source_revision": "ce02f2115b6796592340e475c1650a0d3a8d56a2", "source_ref": {"path": "DLSD/images/val/CANCZ103374-1.jpg"}, "sha256_source": "c6ba5de7c1ab47e2059d3965c2601dde057c83a4d1a716d1d5fa51510c2c9a67", "group": "dlsd:CANCZ103374", "issue_type": "damaged_luggage", "object": "suitcase", "confidence_level": "high", "description": "A cream suitcase has a large broken hole in its upper shell.", "company_reason": "other", "customer_message": "What's the weather like in Lisbon this week?"}
{"id": "cv072", "cid": "c073", "domain": "atlas", "source": "dlsd", "source_kind": "github", "source_repo": "github6-dev/luggage-damage-datasets", "source_revision": "ce02f2115b6796592340e475c1650a0d3a8d56a2", "source_ref": {"path": "DLSD/images/val/CANCZ104967-1.jpg"}, "sha256_source": "25c9b1e8817032fb623ad742deb07fb7130e0461fba89c3c69739b469a2e11c7", "group": "dlsd:CANCZ104967", "issue_type": "damaged_luggage", "object": "suitcase", "confidence_level": "high", "description": "A pale pink ribbed suitcase has long cracks across its shell.", "company_reason": "other", "customer_message": "Do you have an office in Sfax?"}
{"id": "cv073", "cid": "c074", "domain": "atlas", "source": "dlsd", "source_kind": "github", "source_repo": "github6-dev/luggage-damage-datasets", "source_revision": "ce02f2115b6796592340e475c1650a0d3a8d56a2", "source_ref": {"path": "DLSD/images/val/CANCZ105390-1.jpg"}, "sha256_source": "b0c7a63cd58a65e5dd7a5b94940e6be7d4595a49cbdd50da127b9067bba46e35", "group": "dlsd:CANCZ105390", "issue_type": "unclear", "object": "suitcase", "confidence_level": "low", "description": "A close photo of a suitcase mostly covered by airline tags; possible deformation cannot be confirmed.", "company_reason": "other", "customer_message": "Can I pay in instalments?"}
{"id": "cv074", "cid": "c075", "domain": "atlas", "source": "dlsd", "source_kind": "github", "source_repo": "github6-dev/luggage-damage-datasets", "source_revision": "ce02f2115b6796592340e475c1650a0d3a8d56a2", "source_ref": {"path": "DLSD/images/val/CANCZ103809-1.jpg"}, "sha256_source": "c1731a47754ae6c319444c6263686899a00417acc90557734d394d6e860c91c0", "group": "dlsd:CANCZ103809", "issue_type": "damaged_luggage", "object": "suitcase", "confidence_level": "high", "description": "A silver hard-shell suitcase has a deep dent pushed into its side.", "company_reason": null, "customer_message": null}
{"id": "cv075", "cid": "c076", "domain": "atlas", "source": "dlsd", "source_kind": "github", "source_repo": "github6-dev/luggage-damage-datasets", "source_revision": "ce02f2115b6796592340e475c1650a0d3a8d56a2", "source_ref": {"path": "DLSD/images/val/CANCZ104843-1.jpg"}, "sha256_source": "bb03ef38d959ad25ac85596884bdc2a529aa87529ee8b0619b317b803a399b0a", "group": "dlsd:CANCZ104843", "issue_type": "damaged_luggage", "object": "suitcase", "confidence_level": "medium", "description": "A black suitcase has a bent and deformed shell near the handle.", "company_reason": null, "customer_message": null}
{"id": "cv076", "cid": "c077", "domain": "atlas", "source": "dlsd", "source_kind": "github", "source_repo": "github6-dev/luggage-damage-datasets", "source_revision": "ce02f2115b6796592340e475c1650a0d3a8d56a2", "source_ref": {"path": "DLSD/images/val/CANCZ104474-1.jpg"}, "sha256_source": "0a130ad5910700b86c4b9d1cc87f15f4ea48c48d0f610031281c2390c9be3776", "group": "dlsd:CANCZ104474", "issue_type": "damaged_luggage", "object": "suitcase", "confidence_level": "medium", "description": "A grey suitcase with airline tags has a dented, deformed corner.", "company_reason": null, "customer_message": null}
{"id": "cv077", "cid": "c078", "domain": "atlas", "source": "dlsd", "source_kind": "github", "source_repo": "github6-dev/luggage-damage-datasets", "source_revision": "ce02f2115b6796592340e475c1650a0d3a8d56a2", "source_ref": {"path": "DLSD/images/val/BHYCZ000971-1.jpg"}, "sha256_source": "61d13a2fb4fcff6a0236ec9f7e8999cc676c81335950999acfa63664d577e3f4", "group": "dlsd:BHYCZ000971", "issue_type": "damaged_luggage", "object": "suitcase", "confidence_level": "medium", "description": "A grey aluminium suitcase has a bent frame and a dented corner.", "company_reason": null, "customer_message": null}
{"id": "cv078", "cid": "c079", "domain": "atlas", "source": "dlsd", "source_kind": "github", "source_repo": "github6-dev/luggage-damage-datasets", "source_revision": "ce02f2115b6796592340e475c1650a0d3a8d56a2", "source_ref": {"path": "DLSD/images/val/CANCZ102725-1.jpg"}, "sha256_source": "9bb6c6050adf9c4fb85f40cf6695c5a4f43e82b98a2db3925db4f7c5d84d0fe2", "group": "dlsd:CANCZ102725", "issue_type": "damaged_luggage", "object": "suitcase", "confidence_level": "medium", "description": "A yellow hard-shell suitcase shows a crack and scratch lines next to its baggage tag.", "company_reason": null, "customer_message": null}
{"id": "cv079", "cid": "c080", "domain": "atlas", "source": "dlsd", "source_kind": "github", "source_repo": "github6-dev/luggage-damage-datasets", "source_revision": "ce02f2115b6796592340e475c1650a0d3a8d56a2", "source_ref": {"path": "DLSD/images/val/CANCZ104126-1.jpg"}, "sha256_source": "8a48af0cd0bce9a19bcad899f477eea0966fed03d35e0560b166696d3baae250", "group": "dlsd:CANCZ104126", "issue_type": "unclear", "object": "suitcase", "confidence_level": "low", "description": "A grey ribbed suitcase with a baggage tag; no clear damage is visible, but small scratches cannot be ruled out.", "company_reason": null, "customer_message": null}
{"id": "cv080", "cid": "c081", "domain": "atlas", "source": "dlsd", "source_kind": "github", "source_repo": "github6-dev/luggage-damage-datasets", "source_revision": "ce02f2115b6796592340e475c1650a0d3a8d56a2", "source_ref": {"path": "DLSD/images/val/CANCZ101734-1.jpg"}, "sha256_source": "4c9ac68b97dfa9e80513436e7c53f5362f0a9ce01f2a53d80f6e0d6a15619110", "group": "dlsd:CANCZ101734", "issue_type": "unclear", "object": "suitcase", "confidence_level": "low", "description": "A pink suitcase with a heart pattern and a baggage tag; no clear damage is visible, but small scratches cannot be ruled out.", "company_reason": null, "customer_message": null}
{"id": "cv081", "cid": "c082", "domain": "atlas", "source": "olcd", "source_kind": "github", "source_repo": "github6-dev/luggage-damage-datasets", "source_revision": "ce02f2115b6796592340e475c1650a0d3a8d56a2", "source_ref": {"path": "OLCD/Normal/images/val/000001T_002.jpg"}, "sha256_source": "b23e9048d27b32636a1a2e47eb66f01c3bccc8434033f9e0c6d00e0abc07d1b8", "group": "olcd:c082", "issue_type": "no_visible_issue", "object": "suitcase", "confidence_level": "high", "description": "A navy hard-shell suitcase standing upright; no damage is visible.", "company_reason": "booking_status", "customer_message": "What time does my flight leave on Friday?"}
{"id": "cv082", "cid": "c083", "domain": "atlas", "source": "olcd", "source_kind": "github", "source_repo": "github6-dev/luggage-damage-datasets", "source_revision": "ce02f2115b6796592340e475c1650a0d3a8d56a2", "source_ref": {"path": "OLCD/Normal/images/val/010001_361.jpg"}, "sha256_source": "478bc04fdbf4ffce969724bf0a31cfeb9b98f4eca6633f3f18fc44ed93f7b3a4", "group": "olcd:c083", "issue_type": "no_visible_issue", "object": "suitcase", "confidence_level": "high", "description": "A blue hard-shell suitcase standing upright; no damage is visible.", "company_reason": "booking_status", "customer_message": "Is my flight booking 7008 confirmed?"}
{"id": "cv083", "cid": "c084", "domain": "atlas", "source": "olcd", "source_kind": "github", "source_repo": "github6-dev/luggage-damage-datasets", "source_revision": "ce02f2115b6796592340e475c1650a0d3a8d56a2", "source_ref": {"path": "OLCD/Normal/images/val/0375_g_1.jpg"}, "sha256_source": "989cd777284d257a4611939f8897b945abf478cc52d46f4aface3737a36a43de", "group": "olcd:c084", "issue_type": "no_visible_issue", "object": "suitcase", "confidence_level": "high", "description": "A silver suitcase lying on a baggage belt; no damage is visible.", "company_reason": "delay_disruption", "customer_message": "My flight is delayed and nobody tells us anything at the gate."}
{"id": "cv084", "cid": "c085", "domain": "atlas", "source": "olcd", "source_kind": "github", "source_repo": "github6-dev/luggage-damage-datasets", "source_revision": "ce02f2115b6796592340e475c1650a0d3a8d56a2", "source_ref": {"path": "OLCD/Normal/images/val/0730_g_3.jpg"}, "sha256_source": "2287045fb0828975a819c894ea211ffc03d00f0fe939d5cfca7572a618850b63", "group": "olcd:c085", "issue_type": "no_visible_issue", "object": "suitcase", "confidence_level": "high", "description": "A pale grey suitcase lying flat; no damage is visible.", "company_reason": "delay_disruption", "customer_message": "We have been waiting at the airport for three hours, what is going on?"}
{"id": "cv085", "cid": "c086", "domain": "atlas", "source": "olcd", "source_kind": "github", "source_repo": "github6-dev/luggage-damage-datasets", "source_revision": "ce02f2115b6796592340e475c1650a0d3a8d56a2", "source_ref": {"path": "OLCD/Normal/images/val/0848_g_3.jpg"}, "sha256_source": "314dcd53dae47452edb736607b928e0e8074b8755565512f68b45e809ca45c99", "group": "olcd:c086", "issue_type": "no_visible_issue", "object": "suitcase", "confidence_level": "high", "description": "A dark grey suitcase with a baggage tag lying on a belt; no damage is visible.", "company_reason": "delay_disruption", "customer_message": "The plane left four hours late and I missed my connection."}
{"id": "cv086", "cid": "c087", "domain": "atlas", "source": "olcd", "source_kind": "github", "source_repo": "github6-dev/luggage-damage-datasets", "source_revision": "ce02f2115b6796592340e475c1650a0d3a8d56a2", "source_ref": {"path": "OLCD/Normal/images/val/1570_g_3.jpg"}, "sha256_source": "dc75b9b73ac92e8907df07bc6bafcd59054bcfa469ff7edba74a0d439680b75d", "group": "olcd:c087", "issue_type": "no_visible_issue", "object": "suitcase", "confidence_level": "medium", "description": "A gold suitcase lying on a baggage belt; the shell looks intact in this photo.", "company_reason": "delay_disruption", "customer_message": "My flight was delayed and my bag is still not on the belt."}
{"id": "cv087", "cid": "c088", "domain": "atlas", "source": "olcd", "source_kind": "github", "source_repo": "github6-dev/luggage-damage-datasets", "source_revision": "ce02f2115b6796592340e475c1650a0d3a8d56a2", "source_ref": {"path": "OLCD/Normal/images/val/1758_g_2.jpg"}, "sha256_source": "41de6f154e66003c750b5a4bba214a83fd0a84671b521bc4ffa0fbe1d2ea399b", "group": "olcd:c088", "issue_type": "no_visible_issue", "object": "suitcase", "confidence_level": "high", "description": "A navy ribbed suitcase standing on a belt; no damage is visible.", "company_reason": "delay_disruption", "customer_message": "The flight is delayed again, what is happening?"}
{"id": "cv088", "cid": "c089", "domain": "atlas", "source": "olcd", "source_kind": "github", "source_repo": "github6-dev/luggage-damage-datasets", "source_revision": "ce02f2115b6796592340e475c1650a0d3a8d56a2", "source_ref": {"path": "OLCD/Normal/images/val/2342_p_3.jpg"}, "sha256_source": "e4aee532859ebb513380935ca6ac8bb239a1d710d0306708b5fe7d856859c218", "group": "olcd:c089", "issue_type": "no_visible_issue", "object": "suitcase", "confidence_level": "high", "description": "A white suitcase with black trim lying on its side; no damage is visible.", "company_reason": "delay_disruption", "customer_message": "Our departure was pushed back twice tonight."}
{"id": "cv089", "cid": "c090", "domain": "atlas", "source": "olcd", "source_kind": "github", "source_repo": "github6-dev/luggage-damage-datasets", "source_revision": "ce02f2115b6796592340e475c1650a0d3a8d56a2", "source_ref": {"path": "OLCD/Normal/images/val/2569_g_3.jpg"}, "sha256_source": "0b975f5202f14b5ff7d25eb3b38d45d7f84a233469e19ed2fd22eaad9e277cf5", "group": "olcd:c090", "issue_type": "no_visible_issue", "object": "suitcase", "confidence_level": "medium", "description": "A light blue suitcase seen from the top with its handle; no damage is visible.", "company_reason": "delay_disruption", "customer_message": "The flight was late and we waited a long time for the bags."}
{"id": "cv090", "cid": "c091", "domain": "atlas", "source": "olcd", "source_kind": "github", "source_repo": "github6-dev/luggage-damage-datasets", "source_revision": "ce02f2115b6796592340e475c1650a0d3a8d56a2", "source_ref": {"path": "OLCD/Normal/images/val/2787_p_3.jpg"}, "sha256_source": "3c2227f032ff8f876f0f1dd089fc899e66d2ef553fcd6769c08e823182cefcb0", "group": "olcd:c091", "issue_type": "no_visible_issue", "object": "suitcase", "confidence_level": "high", "description": "A gold ribbed suitcase lying flat; no damage is visible.", "company_reason": "delay_disruption", "customer_message": "My flight is delayed by five hours, nobody explains why."}
{"id": "cv091", "cid": "c092", "domain": "atlas", "source": "olcd", "source_kind": "github", "source_repo": "github6-dev/luggage-damage-datasets", "source_revision": "ce02f2115b6796592340e475c1650a0d3a8d56a2", "source_ref": {"path": "OLCD/Normal/images/val/3714_g_2.jpg"}, "sha256_source": "fc5f1fb1f06255c8648549e717d3728f2b64a076e3544344f7230526d586f020", "group": "olcd:c092", "issue_type": "no_visible_issue", "object": "suitcase", "confidence_level": "medium", "description": "A light blue and pink suitcase lying on a belt; no damage is visible.", "company_reason": "delay_disruption", "customer_message": "The plane is still on the ground two hours after departure time."}
{"id": "cv092", "cid": "c093", "domain": "atlas", "source": "olcd", "source_kind": "github", "source_repo": "github6-dev/luggage-damage-datasets", "source_revision": "ce02f2115b6796592340e475c1650a0d3a8d56a2", "source_ref": {"path": "OLCD/Normal/images/val/3950_p_4.jpg"}, "sha256_source": "97f2c184288250c6ff063b0e38735ee251d136ffe351a2aa9d1e82e7c48e8995", "group": "olcd:c093", "issue_type": "no_visible_issue", "object": "suitcase", "confidence_level": "high", "description": "A grey suitcase lying flat with its wheels; no damage is visible.", "company_reason": "delay_disruption", "customer_message": "Our flight was delayed and we arrived in the middle of the night."}
{"id": "cv093", "cid": "c094", "domain": "atlas", "source": "olcd", "source_kind": "github", "source_repo": "github6-dev/luggage-damage-datasets", "source_revision": "ce02f2115b6796592340e475c1650a0d3a8d56a2", "source_ref": {"path": "OLCD/Normal/images/val/行李箱_311.jpg"}, "sha256_source": "e1700c3c0ce2ceae481dd51f452d190013b828b097ccd46ec7d901e887ab14c1", "group": "olcd:c094", "issue_type": "no_visible_issue", "object": "suitcase", "confidence_level": "high", "description": "A turquoise aluminium-frame suitcase standing outdoors; no damage is visible.", "company_reason": "rebooking", "customer_message": "My flight was cancelled, can you put me on the next one?"}
{"id": "cv094", "cid": "c096", "domain": "atlas", "source": "olcd", "source_kind": "github", "source_repo": "github6-dev/luggage-damage-datasets", "source_revision": "ce02f2115b6796592340e475c1650a0d3a8d56a2", "source_ref": {"path": "OLCD/Normal/images/val/行李箱_550.jpg"}, "sha256_source": "7535cb306bcffe0eba0ac6cd51c417865453b182999f64be5ea0e0c03857aa32", "group": "olcd:c096", "issue_type": "no_visible_issue", "object": "suitcase", "confidence_level": "high", "description": "A small grey cabin suitcase standing on a tiled floor; no damage is visible.", "company_reason": "rebooking", "customer_message": "Can you rebook me on a flight tomorrow morning?"}
{"id": "cv095", "cid": "c097", "domain": "atlas", "source": "olcd", "source_kind": "github", "source_repo": "github6-dev/luggage-damage-datasets", "source_revision": "ce02f2115b6796592340e475c1650a0d3a8d56a2", "source_ref": {"path": "OLCD/Normal/images/val/行李箱_747.jpg"}, "sha256_source": "66c122234825556db883b693845f9aa28d7c88cbd378431ff906b2566cc382d7", "group": "olcd:c097", "issue_type": "no_visible_issue", "object": "suitcase", "confidence_level": "high", "description": "A white aluminium suitcase standing on a tiled floor; no damage is visible.", "company_reason": "rebooking", "customer_message": "I'd like to change my flight to a later date."}
{"id": "cv096", "cid": "c098", "domain": "atlas", "source": "hotels50k", "source_kind": "hf_parquet", "source_repo": "alperctnkaya/hotels-50k-test", "source_revision": "bb4ec54cebb50c134702601f61e9f0779f36bafd", "source_ref": {"parquet": "data/unoccluded-00002-of-00003.parquet", "row": 107}, "sha256_source": "3a3077144db52a32c19d5239311e38a530e75907be32f74b464c6331bbadaf66", "group": "hotel:42571", "issue_type": "no_visible_issue", "object": "hotel_room", "confidence_level": "high", "description": "A hotel room with a made king bed, white sheets and bedside lamps; no damage or dirt is visible.", "company_reason": "booking_status", "customer_message": "Is this the room I booked? Can you check my booking 7003?"}
{"id": "cv097", "cid": "c099", "domain": "atlas", "source": "hotels50k", "source_kind": "hf_parquet", "source_repo": "alperctnkaya/hotels-50k-test", "source_revision": "bb4ec54cebb50c134702601f61e9f0779f36bafd", "source_ref": {"parquet": "data/unoccluded-00002-of-00003.parquet", "row": 1113}, "sha256_source": "4b669f60c1d51815a3146455ccf1ef3603de61de4693c950b61c21a802bf4558", "group": "hotel:41631", "issue_type": "no_visible_issue", "object": "hotel_room", "confidence_level": "high", "description": "A hotel room with a made bed and a bench at its foot; the room looks clean.", "company_reason": "booking_status", "customer_message": "Is my hotel reservation confirmed for next week?"}
{"id": "cv098", "cid": "c100", "domain": "atlas", "source": "hotels50k", "source_kind": "hf_parquet", "source_repo": "alperctnkaya/hotels-50k-test", "source_revision": "bb4ec54cebb50c134702601f61e9f0779f36bafd", "source_ref": {"parquet": "data/unoccluded-00002-of-00003.parquet", "row": 1235}, "sha256_source": "f138e2f90bbfb1f9de04b465477e577137023df4925e8d2cbf6605532136eb50", "group": "hotel:80756", "issue_type": "no_visible_issue", "object": "hotel_room", "confidence_level": "high", "description": "A hotel room with a TV dresser, a desk and a window view; no damage or dirt is visible.", "company_reason": "booking_status", "customer_message": "Can you check my booking 7005 for me?"}
{"id": "cv099", "cid": "c101", "domain": "atlas", "source": "hotels50k", "source_kind": "hf_parquet", "source_repo": "alperctnkaya/hotels-50k-test", "source_revision": "bb4ec54cebb50c134702601f61e9f0779f36bafd", "source_ref": {"parquet": "data/unoccluded-00002-of-00003.parquet", "row": 1242}, "sha256_source": "2d0c3b400b88196ffe988f20a43ce49a53612c44efb0abb16e2b1ebcd7afc00d", "group": "hotel:22119", "issue_type": "no_visible_issue", "object": "hotel_room", "confidence_level": "high", "description": "A bright hotel room with a TV, a dresser and a desk; the room looks clean.", "company_reason": "booking_status", "customer_message": "What time is check-in for my reservation?"}
{"id": "cv100", "cid": "c102", "domain": "atlas", "source": "hotels50k", "source_kind": "hf_parquet", "source_repo": "alperctnkaya/hotels-50k-test", "source_revision": "bb4ec54cebb50c134702601f61e9f0779f36bafd", "source_ref": {"parquet": "data/unoccluded-00002-of-00003.parquet", "row": 1336}, "sha256_source": "40d5fcadc17ffa6ebfbe2eee238780ae10344a7bc66b3563851aa54d977e48eb", "group": "hotel:19714", "issue_type": "no_visible_issue", "object": "hotel_room", "confidence_level": "high", "description": "A hotel bed with white sheets and pillows; no damage or dirt is visible.", "company_reason": "booking_status", "customer_message": "Is breakfast included in my booking?"}
{"id": "cv101", "cid": "c103", "domain": "atlas", "source": "hotels50k", "source_kind": "hf_parquet", "source_repo": "alperctnkaya/hotels-50k-test", "source_revision": "bb4ec54cebb50c134702601f61e9f0779f36bafd", "source_ref": {"parquet": "data/unoccluded-00002-of-00003.parquet", "row": 2113}, "sha256_source": "45ecca7a991792bb5182803e74f293d85fa55c7b8a964937a2c97b4652f54a7c", "group": "hotel:46104", "issue_type": "no_visible_issue", "object": "hotel_room", "confidence_level": "high", "description": "A hotel room with a made bed, a desk and an armchair; the room looks tidy.", "company_reason": "booking_status", "customer_message": "Can you confirm my hotel booking 7002?"}
{"id": "cv102", "cid": "c104", "domain": "atlas", "source": "hotels50k", "source_kind": "hf_parquet", "source_repo": "alperctnkaya/hotels-50k-test", "source_revision": "bb4ec54cebb50c134702601f61e9f0779f36bafd", "source_ref": {"parquet": "data/unoccluded-00002-of-00003.parquet", "row": 2187}, "sha256_source": "72159f671ec515908e69610025b799e6131804faa5852e3f279b10b348903ad0", "group": "hotel:95726", "issue_type": "unclear", "object": "hotel_room", "confidence_level": "low", "description": "A used hotel room with an unmade bed, towels on the bed and bottles on the table; cleanliness cannot be judged from this photo.", "company_reason": "other", "customer_message": "Can I get a printed invoice?"}
{"id": "cv103", "cid": "c105", "domain": "atlas", "source": "hotels50k", "source_kind": "hf_parquet", "source_repo": "alperctnkaya/hotels-50k-test", "source_revision": "bb4ec54cebb50c134702601f61e9f0779f36bafd", "source_ref": {"parquet": "data/unoccluded-00002-of-00003.parquet", "row": 2207}, "sha256_source": "6b76f7c684813dd6af13f1f30d11450cc2fd6c8441a6ffcd45588568ad8a8b4b", "group": "hotel:91144", "issue_type": "no_visible_issue", "object": "hotel_room", "confidence_level": "high", "description": "A hotel room with a made bed, a red armchair and a balcony door; the room looks clean.", "company_reason": "booking_status", "customer_message": "How many nights is my booking 7006?"}
{"id": "cv104", "cid": "c106", "domain": "atlas", "source": "hotels50k", "source_kind": "hf_parquet", "source_repo": "alperctnkaya/hotels-50k-test", "source_revision": "bb4ec54cebb50c134702601f61e9f0779f36bafd", "source_ref": {"parquet": "data/unoccluded-00002-of-00003.parquet", "row": 2291}, "sha256_source": "d4e3d02629eca3dfeae4098a27fdd78529e411ac9b4ce45ac0c9e311ffd96aca", "group": "hotel:92395", "issue_type": "no_visible_issue", "object": "hotel_room", "confidence_level": "high", "description": "A hotel room with a made bed, a TV and an armchair by the window; no damage is visible.", "company_reason": "booking_status", "customer_message": "Is my reservation still active?"}
{"id": "cv105", "cid": "c107", "domain": "atlas", "source": "hotels50k", "source_kind": "hf_parquet", "source_repo": "alperctnkaya/hotels-50k-test", "source_revision": "bb4ec54cebb50c134702601f61e9f0779f36bafd", "source_ref": {"parquet": "data/unoccluded-00002-of-00003.parquet", "row": 2334}, "sha256_source": "b6adf8ebe38a9e25ddf712be500c8e3a1f0352f06cd330ef1c2aca3cfe3bdaf4", "group": "hotel:82797", "issue_type": "no_visible_issue", "object": "hotel_room", "confidence_level": "high", "description": "A hotel room with a made king bed and a small kitchen area; the room looks clean.", "company_reason": "cancellation", "customer_message": "My plans changed, how do I cancel my stay?"}
{"id": "cv106", "cid": "c108", "domain": "atlas", "source": "hotels50k", "source_kind": "hf_parquet", "source_repo": "alperctnkaya/hotels-50k-test", "source_revision": "bb4ec54cebb50c134702601f61e9f0779f36bafd", "source_ref": {"parquet": "data/unoccluded-00002-of-00003.parquet", "row": 2407}, "sha256_source": "4a06aabcce33006249ae36701970d17c947ea9ed92a0d1bf310d06dbacb32cfd", "group": "hotel:36661", "issue_type": "no_visible_issue", "object": "hotel_room", "confidence_level": "high", "description": "A hotel bed with white sheets under two framed pictures; no damage or dirt is visible.", "company_reason": "cancellation", "customer_message": "Please cancel my hotel booking 7007."}
{"id": "cv107", "cid": "c109", "domain": "atlas", "source": "hotels50k", "source_kind": "hf_parquet", "source_repo": "alperctnkaya/hotels-50k-test", "source_revision": "bb4ec54cebb50c134702601f61e9f0779f36bafd", "source_ref": {"parquet": "data/unoccluded-00002-of-00003.parquet", "row": 254}, "sha256_source": "5a51db85d20dd5dc052042cf6773cda42d41cc681d262b81a198065578e9086b", "group": "hotel:13767", "issue_type": "no_visible_issue", "object": "hotel_room", "confidence_level": "high", "description": "A hotel room with a made bed with a patterned runner; the room looks tidy.", "company_reason": "cancellation", "customer_message": "I can't travel anymore, please cancel my reservation."}
{"id": "cv108", "cid": "c110", "domain": "atlas", "source": "hotels50k", "source_kind": "hf_parquet", "source_repo": "alperctnkaya/hotels-50k-test", "source_revision": "bb4ec54cebb50c134702601f61e9f0779f36bafd", "source_ref": {"parquet": "data/unoccluded-00002-of-00003.parquet", "row": 2673}, "sha256_source": "f7b94bcebe2038446eef1c570aaf2e589221f6d391c9e68ae0c47d00efb79ba9", "group": "hotel:119875", "issue_type": "no_visible_issue", "object": "hotel_room", "confidence_level": "high", "description": "A hotel room with a made bed with a red runner and red walls; no damage is visible.", "company_reason": "rebooking", "customer_message": "Can you move me to another room or another hotel?"}
{"id": "cv109", "cid": "c111", "domain": "atlas", "source": "hotels50k", "source_kind": "hf_parquet", "source_repo": "alperctnkaya/hotels-50k-test", "source_revision": "bb4ec54cebb50c134702601f61e9f0779f36bafd", "source_ref": {"parquet": "data/unoccluded-00002-of-00003.parquet", "row": 295}, "sha256_source": "a0c4790228c9707b0c613288346b9210d22ab2632081601096dec9b835bbc745", "group": "hotel:79382", "issue_type": "no_visible_issue", "object": "hotel_room", "confidence_level": "high", "description": "A hotel bed with a black and white patterned runner between two lamps; the room looks clean.", "company_reason": "rebooking", "customer_message": "I'd like to change my stay to a later date."}
{"id": "cv110", "cid": "c112", "domain": "atlas", "source": "hotels50k", "source_kind": "hf_parquet", "source_repo": "alperctnkaya/hotels-50k-test", "source_revision": "bb4ec54cebb50c134702601f61e9f0779f36bafd", "source_ref": {"parquet": "data/unoccluded-00002-of-00003.parquet", "row": 3235}, "sha256_source": "8ac63c34e056c5e380b5d14da982088f41ebf0019a408f8e45858948f7104bcc", "group": "hotel:26228", "issue_type": "no_visible_issue", "object": "hotel_room", "confidence_level": "high", "description": "A hotel room with a made bed with a brown runner; no damage or dirt is visible.", "company_reason": "rebooking", "customer_message": "Can you rebook my hotel for next weekend instead?"}
{"id": "cv111", "cid": "c113", "domain": "atlas", "source": "hotels50k", "source_kind": "hf_parquet", "source_repo": "alperctnkaya/hotels-50k-test", "source_revision": "bb4ec54cebb50c134702601f61e9f0779f36bafd", "source_ref": {"parquet": "data/unoccluded-00002-of-00003.parquet", "row": 2115}, "sha256_source": "d4d190b6ff1a65bfd9023e3711edde5dd2253e6a52597c992d1e7990ad00e0e5", "group": "hotel:11760", "issue_type": "no_visible_issue", "object": "bathroom", "confidence_level": "high", "description": "A hotel bathroom with a bathtub, a shower curtain and a toilet; it looks clean.", "company_reason": "special_assistance", "customer_message": "I use a wheelchair, is the bathroom accessible?"}
{"id": "cv112", "cid": "c114", "domain": "atlas", "source": "hotels50k", "source_kind": "hf_parquet", "source_repo": "alperctnkaya/hotels-50k-test", "source_revision": "bb4ec54cebb50c134702601f61e9f0779f36bafd", "source_ref": {"parquet": "data/unoccluded-00002-of-00003.parquet", "row": 234}, "sha256_source": "249d47c4a5eaf0c04cb7e0dd5166266d0c7980330b0b8ca2c20e053d28d622ba", "group": "hotel:84938", "issue_type": "no_visible_issue", "object": "bathroom", "confidence_level": "high", "description": "A hotel bathroom with a toilet, folded towels and a hair dryer; it looks clean.", "company_reason": "special_assistance", "customer_message": "My mother needs a room with grab bars, can you help?"}
{"id": "cv113", "cid": "c115", "domain": "atlas", "source": "hotels50k", "source_kind": "hf_parquet", "source_repo": "alperctnkaya/hotels-50k-test", "source_revision": "bb4ec54cebb50c134702601f61e9f0779f36bafd", "source_ref": {"parquet": "data/unoccluded-00002-of-00003.parquet", "row": 2466}, "sha256_source": "b9b39117c76bc191afa7f298279e1d865884e1c391c7f2754fe716f2dc1e21e3", "group": "hotel:95099", "issue_type": "no_visible_issue", "object": "bathroom", "confidence_level": "high", "description": "A hotel bathroom with a sink, a toilet and stacked towels; it looks clean.", "company_reason": "special_assistance", "customer_message": "I need an accessible bathroom, I use a wheelchair."}
{"id": "cv114", "cid": "c116", "domain": "atlas", "source": "hotels50k", "source_kind": "hf_parquet", "source_repo": "alperctnkaya/hotels-50k-test", "source_revision": "bb4ec54cebb50c134702601f61e9f0779f36bafd", "source_ref": {"parquet": "data/unoccluded-00002-of-00003.parquet", "row": 3163}, "sha256_source": "ff5b4b069b7759e7b3fa14d02af14339d33bb7c8fb04ad529a62df7edfbf6eda", "group": "hotel:24174", "issue_type": "no_visible_issue", "object": "bathroom", "confidence_level": "high", "description": "A hotel bathroom with a toilet and towels on a rack; it looks clean.", "company_reason": "special_assistance", "customer_message": "I'm travelling with a guide dog, what do I need to do?"}
{"id": "cv115", "cid": "c117", "domain": "atlas", "source": "hotels50k", "source_kind": "hf_parquet", "source_repo": "alperctnkaya/hotels-50k-test", "source_revision": "bb4ec54cebb50c134702601f61e9f0779f36bafd", "source_ref": {"parquet": "data/unoccluded-00002-of-00003.parquet", "row": 4387}, "sha256_source": "067e9389c31558fa0937d6cd9eaad32b67f6a434f9773786facc71abe33b66ed", "group": "hotel:73869", "issue_type": "no_visible_issue", "object": "bathroom", "confidence_level": "high", "description": "A hotel bathroom with a toilet, a sink and hanging towels; it looks clean.", "company_reason": "special_assistance", "customer_message": "My father needs help moving around, can the hotel assist?"}
{"id": "cv116", "cid": "c118", "domain": "atlas", "source": "hotels50k", "source_kind": "hf_parquet", "source_repo": "alperctnkaya/hotels-50k-test", "source_revision": "bb4ec54cebb50c134702601f61e9f0779f36bafd", "source_ref": {"parquet": "data/unoccluded-00002-of-00003.parquet", "row": 4581}, "sha256_source": "2f0897e40e2a39015b20689e8b8dc0105c8ae56a7f4327c21454549c463ce123", "group": "hotel:36785", "issue_type": "no_visible_issue", "object": "bathroom", "confidence_level": "high", "description": "A hotel bathroom with a bathtub, a toilet and a granite sink; it looks clean.", "company_reason": "special_assistance", "customer_message": "I need a room adapted for a wheelchair."}
{"id": "cv117", "cid": "c119", "domain": "atlas", "source": "mobilehome", "source_kind": "hf_zip", "source_repo": "beersrobert/mobile-home-interior-inspection", "source_revision": "28154aef24255af32684c10090a446c6c5aeaa4b", "source_ref": {"zip": "singlewide trailer.zip", "member": "singlewide trailer/20260110_141241.jpg"}, "sha256_source": "52ddc796bfcaafd1843bf815cab62c1e7c0c49952c3411be34e5bf07bc6fb844", "group": "mobilehome:1412", "issue_type": "dirty_or_messy_room", "object": "room", "confidence_level": "high", "description": "A bedroom floor covered with clothes, bags and rubbish next to a clothes rack.", "company_reason": "cancellation", "customer_message": "I want to cancel my hotel reservation, the room looks like this."}
{"id": "cv118", "cid": "c120", "domain": "atlas", "source": "mobilehome", "source_kind": "hf_zip", "source_repo": "beersrobert/mobile-home-interior-inspection", "source_revision": "28154aef24255af32684c10090a446c6c5aeaa4b", "source_ref": {"zip": "singlewide trailer.zip", "member": "singlewide trailer/20260110_141243.jpg"}, "sha256_source": "2402dcf5df7f2ab7317d437263a72e95d6ca4a155d34aa37a15bcaa4c3103548", "group": "mobilehome:1412", "issue_type": "dirty_or_messy_room", "object": "room", "confidence_level": "high", "description": "A room floor with stained rugs, scattered clothes, paper and rubbish.", "company_reason": "cancellation", "customer_message": "Please cancel booking 7004, I can't stay in this room."}
{"id": "cv119", "cid": "c121", "domain": "atlas", "source": "mobilehome", "source_kind": "hf_zip", "source_repo": "beersrobert/mobile-home-interior-inspection", "source_revision": "28154aef24255af32684c10090a446c6c5aeaa4b", "source_ref": {"zip": "singlewide trailer.zip", "member": "singlewide trailer/20260110_141253.jpg"}, "sha256_source": "f04364db4f13442a76a10539f000b0467d00fbabaf61e766baa3493395c96021", "group": "mobilehome:1412", "issue_type": "dirty_or_messy_room", "object": "room", "confidence_level": "high", "description": "A dirty carpet with worn rugs and rubbish lying on the floor.", "company_reason": "cancellation", "customer_message": "How do I cancel my stay? This room is not acceptable."}
{"id": "cv120", "cid": "c122", "domain": "atlas", "source": "mobilehome", "source_kind": "hf_zip", "source_repo": "beersrobert/mobile-home-interior-inspection", "source_revision": "28154aef24255af32684c10090a446c6c5aeaa4b", "source_ref": {"zip": "singlewide trailer.zip", "member": "singlewide trailer/20260110_141305.jpg"}, "sha256_source": "26bf8cf420131fdbc29c707a33c4e2a97f0fa4f4246ea6a743652fadb9507e4d", "group": "mobilehome:1413", "issue_type": "dirty_or_messy_room", "object": "room", "confidence_level": "high", "description": "A dark room with a dirty rag rug, a cable and clutter on the floor.", "company_reason": "refund_request", "customer_message": "I want my money back for this room."}
{"id": "cv121", "cid": "c123", "domain": "atlas", "source": "mobilehome", "source_kind": "hf_zip", "source_repo": "beersrobert/mobile-home-interior-inspection", "source_revision": "28154aef24255af32684c10090a446c6c5aeaa4b", "source_ref": {"zip": "singlewide trailer.zip", "member": "singlewide trailer/20260110_141323.jpg"}, "sha256_source": "b3055f2e865dc656a689815ff9a24ae6348ec1cecdbdc33dc029b725c718a34f", "group": "mobilehome:1413", "issue_type": "dirty_or_messy_room", "object": "room", "confidence_level": "medium", "description": "A mattress on the floor with a stained cover and rubbish around it.", "company_reason": "refund_request", "customer_message": "Can I get a refund for booking 7009? The room was dirty."}
{"id": "cv122", "cid": "c124", "domain": "atlas", "source": "mobilehome", "source_kind": "hf_zip", "source_repo": "beersrobert/mobile-home-interior-inspection", "source_revision": "28154aef24255af32684c10090a446c6c5aeaa4b", "source_ref": {"zip": "singlewide trailer.zip", "member": "singlewide trailer/20260110_141325.jpg"}, "sha256_source": "998dbe877f60da9dadd3f3504b720a84bc4b1cbcd532cf70c8d5181ccf6bea87", "group": "mobilehome:1413", "issue_type": "dirty_or_messy_room", "object": "room", "confidence_level": "high", "description": "A dirty floor with worn rugs, a cable and scattered clothes.", "company_reason": "refund_request", "customer_message": "Please refund my stay, this is not what I paid for."}
{"id": "cv123", "cid": "c125", "domain": "atlas", "source": "mobilehome", "source_kind": "hf_zip", "source_repo": "beersrobert/mobile-home-interior-inspection", "source_revision": "28154aef24255af32684c10090a446c6c5aeaa4b", "source_ref": {"zip": "singlewide trailer.zip", "member": "singlewide trailer/20260110_141330.jpg"}, "sha256_source": "75e8b6e44baac448fc5411358f7a8738bc922831744bee105274dc2f238557a5", "group": "mobilehome:1413", "issue_type": "dirty_or_messy_room", "object": "room", "confidence_level": "medium", "description": "A cluttered bedroom corner with a soiled rug, a towel and cables on the floor.", "company_reason": "refund_request", "customer_message": "I want a refund for the nights I spent in this room."}
{"id": "cv124", "cid": "c126", "domain": "atlas", "source": "mobilehome", "source_kind": "hf_zip", "source_repo": "beersrobert/mobile-home-interior-inspection", "source_revision": "28154aef24255af32684c10090a446c6c5aeaa4b", "source_ref": {"zip": "singlewide trailer.zip", "member": "singlewide trailer/20260110_141447.jpg"}, "sha256_source": "9364e25723ffc9224b675fb3bda1c7891bc3d1a9e941fafab076b70648d4291b", "group": "mobilehome:1414", "issue_type": "dirty_or_messy_room", "object": "room", "confidence_level": "medium", "description": "A crowded clothes rack and shelves with a laundry basket and clutter on the floor.", "company_reason": "rebooking", "customer_message": "I want to change my hotel, please rebook me somewhere else."}
{"id": "cv125", "cid": "c127", "domain": "atlas", "source": "mobilehome", "source_kind": "hf_zip", "source_repo": "beersrobert/mobile-home-interior-inspection", "source_revision": "28154aef24255af32684c10090a446c6c5aeaa4b", "source_ref": {"zip": "singlewide trailer.zip", "member": "singlewide trailer/20260110_141449.jpg"}, "sha256_source": "10f820dcfcd8df61183db0da8d6a277566a7edb2f75e1b1f6a34ffda0db1bb39", "group": "mobilehome:1414", "issue_type": "dirty_or_messy_room", "object": "room", "confidence_level": "high", "description": "A bedroom floor covered with clothes and rubbish below a clothes rack.", "company_reason": "rebooking", "customer_message": "Please move my reservation to another hotel."}
{"id": "cv126", "cid": "c128", "domain": "atlas", "source": "mobilehome", "source_kind": "hf_zip", "source_repo": "beersrobert/mobile-home-interior-inspection", "source_revision": "28154aef24255af32684c10090a446c6c5aeaa4b", "source_ref": {"zip": "singlewide trailer.zip", "member": "singlewide trailer/20260110_142601.jpg"}, "sha256_source": "b2b11d5ca42bb7c45b5a66157df650594f754761f135c32a57ed24b3ffb6e755", "group": "mobilehome:1426", "issue_type": "dirty_or_messy_room", "object": "room", "confidence_level": "medium", "description": "A living area with a cluttered sofa, bags and items left on the floor.", "company_reason": "special_assistance", "customer_message": "I need help, I'm in a wheelchair and the room is not accessible."}
{"id": "cv127", "cid": "c129", "domain": "atlas", "source": "mobilehome", "source_kind": "hf_zip", "source_repo": "beersrobert/mobile-home-interior-inspection", "source_revision": "28154aef24255af32684c10090a446c6c5aeaa4b", "source_ref": {"zip": "singlewide trailer.zip", "member": "singlewide trailer/20260110_142602.jpg"}, "sha256_source": "8c62e02bb747543ea334c932a37bbfc4f164028775cc0d682a7564edb37ece6c", "group": "mobilehome:1426", "issue_type": "dirty_or_messy_room", "object": "room", "confidence_level": "medium", "description": "A living area with a worn rug and clothes piled on a sofa.", "company_reason": "special_assistance", "customer_message": "My son is disabled and needs assistance at the hotel."}
{"id": "cv128", "cid": "c130", "domain": "atlas", "source": "mobilehome", "source_kind": "hf_zip", "source_repo": "beersrobert/mobile-home-interior-inspection", "source_revision": "28154aef24255af32684c10090a446c6c5aeaa4b", "source_ref": {"zip": "singlewide trailer.zip", "member": "singlewide trailer/20260110_142503.jpg"}, "sha256_source": "13e35025767d6c8d4e0eb237430cade9e13b790a79074d82e0ba16a5987955f7", "group": "mobilehome:1425", "issue_type": "damaged_room_fixture", "object": "ceiling", "confidence_level": "high", "description": "A ceiling with a long water stain and peeling patches along a seam.", "company_reason": "cancellation", "customer_message": "I want to cancel the last two nights, the ceiling is damaged."}
{"id": "cv129", "cid": "c131", "domain": "atlas", "source": "mobilehome", "source_kind": "hf_zip", "source_repo": "beersrobert/mobile-home-interior-inspection", "source_revision": "28154aef24255af32684c10090a446c6c5aeaa4b", "source_ref": {"zip": "singlewide trailer.zip", "member": "singlewide trailer/20260110_142504.jpg"}, "sha256_source": "6296ebf47bf39abae29954045058efe8977781dcb625054a41b146bd5d2c662b", "group": "mobilehome:1425", "issue_type": "damaged_room_fixture", "object": "ceiling", "confidence_level": "high", "description": "A ceiling with a long brown water-damage stain along a panel seam near the light.", "company_reason": "refund_request", "customer_message": "Please refund me, the room had water damage on the ceiling."}
{"id": "cv130", "cid": "c132", "domain": "atlas", "source": "mobilehome", "source_kind": "hf_zip", "source_repo": "beersrobert/mobile-home-interior-inspection", "source_revision": "28154aef24255af32684c10090a446c6c5aeaa4b", "source_ref": {"zip": "singlewide trailer.zip", "member": "singlewide trailer/20260110_142506.jpg"}, "sha256_source": "bce7054bce8121dc8ad9246bb859538c6419ee3cf88bee4d4960dc5af54d5deb", "group": "mobilehome:1425", "issue_type": "damaged_room_fixture", "object": "ceiling", "confidence_level": "high", "description": "A ceiling with a water-damaged, peeling strip and stains around a light fitting.", "company_reason": "refund_request", "customer_message": "I want my money back for the trip."}
{"id": "cv131", "cid": "c133", "domain": "atlas", "source": "mobilehome", "source_kind": "hf_zip", "source_repo": "beersrobert/mobile-home-interior-inspection", "source_revision": "28154aef24255af32684c10090a446c6c5aeaa4b", "source_ref": {"zip": "singlewide trailer.zip", "member": "singlewide trailer/20260110_142507.jpg"}, "sha256_source": "e7c7e87aa4a95486e9b9ac63bc3e4dd5812afe5176941efcb5e477fee892d0b0", "group": "mobilehome:1425", "issue_type": "damaged_room_fixture", "object": "ceiling", "confidence_level": "medium", "description": "A ceiling with a stained, damaged seam above a hallway.", "company_reason": "other", "customer_message": "Do you offer airport transfers?"}
{"id": "cv132", "cid": "c134", "domain": "atlas", "source": "mobilehome", "source_kind": "hf_zip", "source_repo": "beersrobert/mobile-home-interior-inspection", "source_revision": "28154aef24255af32684c10090a446c6c5aeaa4b", "source_ref": {"zip": "singlewide trailer.zip", "member": "singlewide trailer/20260110_142536.jpg"}, "sha256_source": "b26575ac142b2ef9a23aeee8f4dfb9ea7c451c7f40abc0793f264cdcf2aebf53", "group": "mobilehome:1425", "issue_type": "unclear", "object": "ceiling", "confidence_level": "low", "description": "A dark photo of a ceiling with faint marks; water damage cannot be confirmed.", "company_reason": "other", "customer_message": "Do you have a newsletter?"}
{"id": "cv133", "cid": "c135", "domain": "atlas", "source": "mobilehome", "source_kind": "hf_zip", "source_repo": "beersrobert/mobile-home-interior-inspection", "source_revision": "28154aef24255af32684c10090a446c6c5aeaa4b", "source_ref": {"zip": "singlewide trailer.zip", "member": "singlewide trailer/20260110_142541.jpg"}, "sha256_source": "d2da78a12baa7fb04872016333498b04eac404414d70fb5c7a737315b96e4c4c", "group": "mobilehome:1425", "issue_type": "unclear", "object": "ceiling", "confidence_level": "low", "description": "A dark photo of a ceiling with faint marks; water damage cannot be confirmed.", "company_reason": null, "customer_message": null}
{"id": "cv134", "cid": "c136", "domain": "atlas", "source": "mobilehome", "source_kind": "hf_zip", "source_repo": "beersrobert/mobile-home-interior-inspection", "source_revision": "28154aef24255af32684c10090a446c6c5aeaa4b", "source_ref": {"zip": "singlewide trailer.zip", "member": "singlewide trailer/20260110_142327.jpg"}, "sha256_source": "a3cd3fcd5cc29f74965259ab0f353de927d20a6c14435f837778f3ed66a9ab9d", "group": "mobilehome:1423", "issue_type": "dirty_or_messy_room", "object": "bathroom", "confidence_level": "high", "description": "A toilet with a stained, discoloured bowl next to a metal rack.", "company_reason": "cancellation", "customer_message": "Please cancel my hotel booking, the bathroom is disgusting."}
{"id": "cv135", "cid": "c137", "domain": "atlas", "source": "mobilehome", "source_kind": "hf_zip", "source_repo": "beersrobert/mobile-home-interior-inspection", "source_revision": "28154aef24255af32684c10090a446c6c5aeaa4b", "source_ref": {"zip": "singlewide trailer.zip", "member": "singlewide trailer/20260110_142330.jpg"}, "sha256_source": "dc81bf8dd8dfa20744bec52248981824a4d643a7af2f9f8f55303317ddf5e55d", "group": "mobilehome:1423", "issue_type": "dirty_or_messy_room", "object": "bathroom", "confidence_level": "medium", "description": "A bathtub with dirt marks and items left on its edge.", "company_reason": "refund_request", "customer_message": "I want a refund, the bathroom was not cleaned."}
{"id": "cv136", "cid": "c138", "domain": "atlas", "source": "mobilehome", "source_kind": "hf_zip", "source_repo": "beersrobert/mobile-home-interior-inspection", "source_revision": "28154aef24255af32684c10090a446c6c5aeaa4b", "source_ref": {"zip": "singlewide trailer.zip", "member": "singlewide trailer/20260110_142331.jpg"}, "sha256_source": "ea0ff741f7fac991907150ff025b19b3afd7e848065ee610f571f6f4a2c43299", "group": "mobilehome:1423", "issue_type": "unclear", "object": "bathroom", "confidence_level": "low", "description": "A small bathroom with a tub and a shower chair; the walls look worn, but damage or dirt is not clear.", "company_reason": null, "customer_message": null}
{"id": "cv137", "cid": "c139", "domain": "atlas", "source": "mobilehome", "source_kind": "hf_zip", "source_repo": "beersrobert/mobile-home-interior-inspection", "source_revision": "28154aef24255af32684c10090a446c6c5aeaa4b", "source_ref": {"zip": "singlewide trailer.zip", "member": "singlewide trailer/20260110_142424.jpg"}, "sha256_source": "ab25ad2a2d51be89913e6aaf28f95d6ba308ed627252da2857ddfe3ed96587f2", "group": "mobilehome:1424", "issue_type": "unclear", "object": "bathroom", "confidence_level": "low", "description": "A toilet and floor in a bathroom corner; the photo is too dark and close to judge cleanliness.", "company_reason": null, "customer_message": null}
{"id": "cv138", "cid": "c140", "domain": "atlas", "source": "mobilehome", "source_kind": "hf_zip", "source_repo": "beersrobert/mobile-home-interior-inspection", "source_revision": "28154aef24255af32684c10090a446c6c5aeaa4b", "source_ref": {"zip": "singlewide trailer.zip", "member": "singlewide trailer/20260110_142435.jpg"}, "sha256_source": "a36e688d52037681fb853e0a8c8a72cb21f108129f821c9e443d28743b1c589f", "group": "mobilehome:1424", "issue_type": "dirty_or_messy_room", "object": "bathroom", "confidence_level": "medium", "description": "A bathroom sink area crowded with bottles and toiletries; the counter looks unclean.", "company_reason": "refund_request", "customer_message": "Please give me my money back for this hotel."}
{"id": "cv139", "cid": "c141", "domain": "atlas", "source": "walldamage", "source_kind": "hf", "source_repo": "LibreYOLO/wall-damage", "source_revision": "bcc219bc9e28b85f9c19249c6736425e9d9ce450", "source_ref": {"path": "test/images/image420_jpg.rf.bc2a9b5daf354d94618f3924cccd8040.jpg"}, "sha256_source": "dc9384d79eac6d6bc6b74c510de629a11139d25da501736082778bb1ebde5fb7", "group": "walldamage:c141", "issue_type": "damaged_room_fixture", "object": "room_wall", "confidence_level": "high", "description": "A white wall above a window frame has a long, wide crack.", "company_reason": "cancellation", "customer_message": "I want to cancel the rest of my booking, the wall in my room is cracked."}
{"id": "cv140", "cid": "c142", "domain": "atlas", "source": "walldamage", "source_kind": "hf", "source_repo": "LibreYOLO/wall-damage", "source_revision": "bcc219bc9e28b85f9c19249c6736425e9d9ce450", "source_ref": {"path": "test/images/image430_jpg.rf.7c62416a3f9c3d0d073be2ff371f355a.jpg"}, "sha256_source": "a65d5a991d6e0f2ef9bf9c80492208b43e63cdb669abdfd5250acdea10244119", "group": "walldamage:c142", "issue_type": "damaged_room_fixture", "object": "room_wall", "confidence_level": "high", "description": "A wall above a window has a long horizontal crack.", "company_reason": "cancellation", "customer_message": "Cancel my reservation please, the room is in bad shape."}
{"id": "cv141", "cid": "c143", "domain": "atlas", "source": "walldamage", "source_kind": "hf", "source_repo": "LibreYOLO/wall-damage", "source_revision": "bcc219bc9e28b85f9c19249c6736425e9d9ce450", "source_ref": {"path": "test/images/image446_jpg.rf.1200a8db51794df948703e237ac1ab01.jpg"}, "sha256_source": "3b52e083031b9851e606637ec710cebe97ad40e3d08c6216a7107bc289865bfd", "group": "walldamage:c143", "issue_type": "damaged_room_fixture", "object": "room_wall", "confidence_level": "high", "description": "A wall corner near a doorway has a jagged crack.", "company_reason": "refund_request", "customer_message": "Can I get a refund for booking 7010?"}
{"id": "cv142", "cid": "c144", "domain": "atlas", "source": "walldamage", "source_kind": "hf", "source_repo": "LibreYOLO/wall-damage", "source_revision": "bcc219bc9e28b85f9c19249c6736425e9d9ce450", "source_ref": {"path": "test/images/image473_jpg.rf.f3696493d719431ed4db8b7ba417f577.jpg"}, "sha256_source": "2f6db463f13bf9bd8d55171e319109945e0ee89c0189ede1af4574dc30586c19", "group": "walldamage:c144", "issue_type": "damaged_room_fixture", "object": "room_wall", "confidence_level": "high", "description": "A wall above a purple door frame has a long horizontal crack.", "company_reason": "refund_request", "customer_message": "Refund my hotel stay please, the room was in bad condition."}
{"id": "cv143", "cid": "c145", "domain": "atlas", "source": "walldamage", "source_kind": "hf", "source_repo": "LibreYOLO/wall-damage", "source_revision": "bcc219bc9e28b85f9c19249c6736425e9d9ce450", "source_ref": {"path": "test/images/image477_jpg.rf.81f0a06c04532372a04d09d534202622.jpg"}, "sha256_source": "c974e1c37d7c52cfb959d1c92d381e9682a3f90275f77c563a8c36b2d54c6eda", "group": "wall:same_doorway_two_crops", "issue_type": "damaged_room_fixture", "object": "room_wall", "confidence_level": "high", "description": "The wall above a doorway is broken, with plaster fallen away and the bricks showing.", "company_reason": "rebooking", "customer_message": "The room is damaged, please rebook me in another hotel."}
{"id": "cv144", "cid": "c146", "domain": "atlas", "source": "walldamage", "source_kind": "hf", "source_repo": "LibreYOLO/wall-damage", "source_revision": "bcc219bc9e28b85f9c19249c6736425e9d9ce450", "source_ref": {"path": "test/images/image501_jpg.rf.33150a33f542dd8857671d0eda6f2faa.jpg"}, "sha256_source": "9d4ba5db9b0cc88620df0b5ec3fce3b0bd5a2b8f00599c19c49fca8f54f57e72", "group": "walldamage:c146", "issue_type": "damaged_room_fixture", "object": "room_wall", "confidence_level": "medium", "description": "A white wall with a thin diagonal crack next to a small blackboard.", "company_reason": "rebooking", "customer_message": "Can you change my booking to a different room?"}
{"id": "cv145", "cid": "c147", "domain": "atlas", "source": "walldamage", "source_kind": "hf", "source_repo": "LibreYOLO/wall-damage", "source_revision": "bcc219bc9e28b85f9c19249c6736425e9d9ce450", "source_ref": {"path": "test/images/image51_jpg.rf.534b524c135cfa9dce0c778968388da3.jpg"}, "sha256_source": "fae18cfed5c473dcad09f6b2857bb2f4797108041589ca46ecda97b1ef16ac53", "group": "walldamage:c147", "issue_type": "damaged_room_fixture", "object": "room_wall", "confidence_level": "high", "description": "A wall and ceiling corner with a broken, cracked edge.", "company_reason": "other", "customer_message": "Do you organise tours in the desert?"}
{"id": "cv146", "cid": "c148", "domain": "atlas", "source": "walldamage", "source_kind": "hf", "source_repo": "LibreYOLO/wall-damage", "source_revision": "bcc219bc9e28b85f9c19249c6736425e9d9ce450", "source_ref": {"path": "valid/images/image482_jpg.rf.55d2166396b0e12aa82a4eb2ef3c909a.jpg"}, "sha256_source": "ecd327d35f31148f7f9e1340fe8e94fef48a451712effbe1b7c0969fcbf3c8e6", "group": "walldamage:c148", "issue_type": "damaged_room_fixture", "object": "room_wall", "confidence_level": "high", "description": "A wall above a door frame has a long diagonal crack.", "company_reason": "other", "customer_message": "How do I contact your agency by phone?"}
{"id": "cv147", "cid": "c149", "domain": "atlas", "source": "walldamage", "source_kind": "hf", "source_repo": "LibreYOLO/wall-damage", "source_revision": "bcc219bc9e28b85f9c19249c6736425e9d9ce450", "source_ref": {"path": "valid/images/image485_jpg.rf.4bf117729185554c8c71ce2db8719dd4.jpg"}, "sha256_source": "9b4385013b9194acab3026a12eec1041ab622a78c700e00d3356ed4d1f918e02", "group": "walldamage:c149", "issue_type": "damaged_room_fixture", "object": "room_wall", "confidence_level": "high", "description": "A wall above a door has several long cracks.", "company_reason": null, "customer_message": null}
{"id": "cv148", "cid": "c150", "domain": "atlas", "source": "walldamage", "source_kind": "hf", "source_repo": "LibreYOLO/wall-damage", "source_revision": "bcc219bc9e28b85f9c19249c6736425e9d9ce450", "source_ref": {"path": "valid/images/image652_jpg.rf.c48e3a722376266ab9901a1129becf04.jpg"}, "sha256_source": "d554cf29aa04d38c750e47f95b3a38716a4393c4e8d8fa2a0d1b58725a45e95c", "group": "walldamage:c150", "issue_type": "damaged_room_fixture", "object": "room_wall", "confidence_level": "high", "description": "A close view of a wall with two crossing cracks.", "company_reason": null, "customer_message": null}
{"id": "cv149", "cid": "c151", "domain": "atlas", "source": "walldamage", "source_kind": "hf", "source_repo": "LibreYOLO/wall-damage", "source_revision": "bcc219bc9e28b85f9c19249c6736425e9d9ce450", "source_ref": {"path": "valid/images/image667_jpg.rf.3af237a5da26466b25cfe51b7b72e35e.jpg"}, "sha256_source": "487bd2eb3d8e4725c08a6bf46cc4a11801df7281eb756cc54ac8b00902468823", "group": "wall:same_doorway_two_crops", "issue_type": "damaged_room_fixture", "object": "room_wall", "confidence_level": "high", "description": "The wall above a doorway is broken open, with plaster missing and rough material showing.", "company_reason": null, "customer_message": null}
{"id": "cv150", "cid": "c152", "domain": "atlas", "source": "walldamage", "source_kind": "hf", "source_repo": "LibreYOLO/wall-damage", "source_revision": "bcc219bc9e28b85f9c19249c6736425e9d9ce450", "source_ref": {"path": "valid/images/image685_jpg.rf.ce04aad6add46b8386795439795ece77.jpg"}, "sha256_source": "f149e5218407f4aa3641b0924de567d85bd0156f78fc02e5ad5bef7423de75cf", "group": "walldamage:c152", "issue_type": "damaged_room_fixture", "object": "room_wall", "confidence_level": "high", "description": "The wall above a corridor opening is badly cracked, with plaster broken off.", "company_reason": null, "customer_message": null}

## Cell 3 — Dataset preparation
Uses an attached Kaggle dataset if one contains `dataset.jsonl`; otherwise downloads the 150 photos from their
pinned public sources (Hugging Face + GitHub, ~1.3 GB, about 2-4 minutes) and checks each photo's sha256.

In [ ]:
import glob, collections, random
import numpy as np
sys.path.insert(0, WORK)
import cv_vision_lib as L
import build_dataset as B
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.benchmark = False

attached = sorted(glob.glob("/kaggle/input/**/dataset.jsonl", recursive=True))
if attached:
    DATA = os.path.dirname(attached[0])
    print("using attached dataset:", DATA)
    summary = json.load(open(os.path.join(DATA, "dataset_summary.json")))
else:
    DATA = os.path.join(WORK, "callverse_vision_dataset")
    try:
        summary = B.build("manifest.jsonl", DATA, os.path.join(WORK, "source_cache"))
    except OSError as e:
        raise SystemExit(f"Download failed ({e}). In Kaggle turn Settings -> Internet ON, then Run All again.")
rows = L.load_rows(DATA)
# checks: required fields, files, decodable images, label values
need = {"id", "image", "domain", "group", "split", "target"}
for r in rows:
    miss = need - set(r)
    if miss:
        raise ValueError(f"{r.get('id')}: missing fields {miss}")
    if not os.path.isfile(r["image_path"]):
        raise FileNotFoundError(r["image_path"])
    L.open_image(r["image_path"])
    ok, why = L.parse_output(L.answer_text(r["target"]))
    if ok is None:
        raise ValueError(f"{r['id']}: target is not a valid answer ({why})")
split = {s: [r for r in rows if r["split"] == s] for s in ("train", "val", "test")}
# leakage: a group (same suitcase / box / hotel / room) must sit in one split only
g2s = collections.defaultdict(set)
for r in rows:
    g2s[r["group"]].add(r["split"])
leaks = [g for g, s in g2s.items() if len(s) > 1]
assert not leaks, f"leakage: {leaks}"
# near-duplicate check ACROSS splits (difference hash on the stored photos)
from PIL import Image
dh = {r["id"]: B.dhash(Image.open(r["image_path"])) for r in rows}
cross = min((B.hamming(dh[a["id"]], dh[b["id"]]), a["id"], b["id"])
            for a in rows for b in rows if a["split"] < b["split"])
print("closest photo pair across splits (dHash distance, 0 = identical):", cross)
print(json.dumps({s: len(v) for s, v in split.items()}), "photos")
tab = collections.Counter((r["target"]["issue_type"], r["split"]) for r in rows)
print(f"{'issue_type':24s} train  val test")
for c in L.ISSUE_TYPES:
    print(f"{c:24s} {tab[(c,'train')]:5d} {tab[(c,'val')]:4d} {tab[(c,'test')]:4d}")
json.dump(summary, open(os.path.join(WORK, "dataset_summary.json"), "w"), indent=1)

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
show = [r for r in (next((r for r in split["train"] if r["target"]["issue_type"] == c), None) for c in L.ISSUE_TYPES) if r]
fig, ax = plt.subplots(2, 4, figsize=(16, 8))
for a, r in zip(ax.ravel(), show):
    a.imshow(L.open_image(r["image_path"])); a.set_title(f"{r['id']} {r['target']['issue_type']}", fontsize=9); a.axis("off")
plt.tight_layout(); plt.savefig("sample_images.png", dpi=80); plt.show()

## Cell 4 — Baseline: original Qwen3-VL-2B (before fine-tuning)
Loaded exactly as it will be for training (4-bit NF4, same image size, greedy decoding), so the comparison is fair.
Two prompts: the original short prompt (style of Molka's benchmark) and the improved prompt (allowed values +
grounding rules). Predictions on **val** and **test** are saved; test numbers are only *reported* in Cell 8 and
never used to choose anything.

In [ ]:
BASE_MODEL = "Qwen/Qwen3-VL-2B-Instruct"
BASE_REVISION = "89644892e4d85e24eaac8bacfd4f463576704203"   # pinned Hugging Face commit
MAX_PIXELS = 512 * 512      # at most 256 image tokens per photo (16-px patches, 2x2 merge)
MIN_PIXELS = 128 * 128
COMPUTE = torch.bfloat16 if BF16 else torch.float16
QUANT = "nf4"               # set to "none" automatically if bitsandbytes 4-bit is not usable on this GPU

processor = AutoProcessor.from_pretrained(BASE_MODEL, revision=BASE_REVISION)
processor.image_processor.size = {"longest_edge": MAX_PIXELS, "shortest_edge": MIN_PIXELS}

def load_base(compute=None):
    """Base model in 4-bit NF4 (vision encoder + lm_head kept un-quantised for image fidelity)."""
    global QUANT
    compute = compute or COMPUTE
    if QUANT == "nf4":
        try:
            q = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4", bnb_4bit_use_double_quant=True,
                                   bnb_4bit_compute_dtype=compute, llm_int8_skip_modules=["visual", "lm_head"])
            m = Qwen3VLForConditionalGeneration.from_pretrained(BASE_MODEL, revision=BASE_REVISION,
                                                                quantization_config=q, dtype=compute, device_map={"": 0})
            return m
        except Exception as e:
            print("4-bit loading failed, falling back to 16-bit LoRA:", repr(e)[:300])
            QUANT = "none"
    return Qwen3VLForConditionalGeneration.from_pretrained(BASE_MODEL, revision=BASE_REVISION, dtype=compute,
                                                           device_map={"": 0})

L.AMP_DTYPE = COMPUTE
base = load_base()
base.eval()
print("quantisation:", QUANT, "| compute dtype:", COMPUTE, "| GPU memory after load:",
      round(torch.cuda.memory_allocated() / 2**30, 2), "GB")
# fp16 safety check on 3 photos: if the model gives NaN/inf losses in fp16, switch to fp32 compute.
probe = [float(L.eval_loss(base, processor, [r], L.PROMPT_IMPROVED)) for r in split["train"][:3]]
if not all(np.isfinite(probe)):
    print("non-finite loss in", COMPUTE, "-> reloading with float32 compute")
    COMPUTE = torch.float32; L.AMP_DTYPE = None
    del base; torch.cuda.empty_cache(); base = load_base(torch.float32); base.eval()
print("probe losses:", [round(x, 3) for x in probe])
env.update({"quantisation": QUANT, "compute_dtype": str(COMPUTE)})

PRED = {}   # (system, split) -> predictions
for system, prompt in (("base_original_prompt", L.PROMPT_BASE), ("base_improved_prompt", L.PROMPT_IMPROVED)):
    for s in ("val", "test"):
        print(f"{system} on {s}:")
        PRED[(system, s)] = L.predict(base, processor, split[s], prompt)
BASE_VAL = {sy: L.metrics(split["val"], PRED[(sy, "val")]) for sy in ("base_original_prompt", "base_improved_prompt")}
for sy, m in BASE_VAL.items():
    print(f"VAL {sy}: accuracy {m['accuracy']:.3f}  macro-F1 {m['macro_f1']:.3f}  damage recall "
          f"{m['damage_recall']:.3f}  JSON valid {m['json_validity']:.3f}  latency {m['latency_s']['mean']:.2f} s")

## Cell 5 — QLoRA configuration
LoRA only on the language model's linear layers (q/k/v/o + MLP). The vision encoder stays frozen: with ~100
training photos, training it would mostly over-fit. The image still drives the update: the LoRA layers read the
image tokens, and the check below compares the loss with the right photo vs. a swapped photo.

In [ ]:
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
VRAM = env["gpu_vram_gb"]
EFFECTIVE_BATCH = 8
CFG_BASE = {"r": 16, "alpha": 32, "dropout": 0.05, "lr": 1e-4, "epochs": 3, "grad_accum": EFFECTIVE_BATCH,
            "batch_size": 1, "seed": SEED, "patience": 2, "prompt": L.PROMPT_IMPROVED,
            "max_pixels": MAX_PIXELS, "quantisation": QUANT, "compute_dtype": str(COMPUTE),
            "gradient_checkpointing": True, "target_modules": L.lora_target_regex()}
print("batch 1 x gradient accumulation", EFFECTIVE_BATCH, "(VRAM", VRAM, "GB)")

def make_peft(cfg):
    torch.manual_seed(cfg["seed"])
    m = load_base(COMPUTE)
    m.config.use_cache = False
    if QUANT == "nf4":
        m = prepare_model_for_kbit_training(m, use_gradient_checkpointing=True,
                                            gradient_checkpointing_kwargs={"use_reentrant": False})
    else:
        m.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})
        m.enable_input_require_grads()
    lc = LoraConfig(r=cfg["r"], lora_alpha=cfg["alpha"], lora_dropout=cfg["dropout"],
                    target_modules=cfg["target_modules"], bias="none", task_type="CAUSAL_LM")
    pm = get_peft_model(m, lc)
    for p in pm.parameters():          # LoRA weights in float32 (needed by the fp16 GradScaler, more stable)
        if p.requires_grad:
            p.data = p.data.float()
    return pm

del base; torch.cuda.empty_cache()
model = make_peft(CFG_BASE)
model.print_trainable_parameters()
names = [n for n, p in model.named_parameters() if p.requires_grad]
assert names and not any("visual" in n for n in names), "LoRA must sit on the language model only"
# genuine image-conditioned update: image tokens present, gradients flow, the image changes the loss
r0 = split["train"][0]
b0 = L.encode_for_training(processor, L.open_image(r0["image_path"]), L.PROMPT_IMPROVED, L.answer_text(r0["target"])).to(0)
n_img = L.count_image_tokens(processor, b0["input_ids"]); n_ans = int((b0["labels"] != -100).sum())
assert n_img > 0 and 0 < n_ans < b0["input_ids"].shape[1]
model.train()
with L._autocast():
    loss0 = model(**b0).loss
loss0.backward()
gsum = sum(float(p.grad.abs().sum()) for n, p in model.named_parameters() if p.requires_grad and p.grad is not None)
model.zero_grad(set_to_none=True)
assert np.isfinite(gsum) and gsum > 0, "no gradient reached the LoRA weights"
probe_rows = split["val"][:8]
CHECK = {"image_tokens_example": n_img, "answer_tokens_example": n_ans, "first_loss": loss0.item(), "grad_abs_sum": gsum,
         "val8_loss_true_images_before": L.eval_loss(model, processor, probe_rows, L.PROMPT_IMPROVED),
         "val8_loss_swapped_images_before": L.eval_loss(model, processor, probe_rows, L.PROMPT_IMPROVED, shuffle_images=True)}
print(json.dumps(CHECK, indent=1))

## Cell 6 — QLoRA training (main configuration)
Batch 1 × 8 accumulation, cosine schedule, answer-only loss, validation loss + validation generation metrics after
every epoch, adapter saved every epoch (`ckpt/<run>/epoch_k`, resume-safe), early stop after 2 epochs without a
better validation loss. The best epoch is chosen on **validation macro-F1** (tie: lower validation loss).

In [ ]:
RUNS = {}
def run_config(name, cfg):
    global model
    out_dir = os.path.join(WORK, "ckpt", name)
    def on_epoch(ep, ep_dir):
        preds = L.predict(model, processor, split["val"], cfg["prompt"])
        m = L.metrics(split["val"], preds, n_boot=200)
        torch.cuda.empty_cache()
        print(f"    val accuracy {m['accuracy']:.3f}  macro-F1 {m['macro_f1']:.3f}  damage recall {m['damage_recall']:.3f}"
              f"  JSON valid {m['json_validity']:.3f}")
        return {"val_accuracy": m["accuracy"], "val_macro_f1": m["macro_f1"], "val_damage_recall": m["damage_recall"],
                "val_json_validity": m["json_validity"], "adapter_dir": ep_dir}
    t0 = time.time()
    hist = L.train_run(model, processor, split["train"], split["val"], cfg, out_dir,
                       use_scaler=(COMPUTE == torch.float16), epoch_callback=on_epoch)
    best = max(hist, key=lambda h: (h["val_macro_f1"], -h["val_loss"]))
    RUNS[name] = {"config": {k: v for k, v in cfg.items() if k != "prompt"}, "history": hist,
                  "best_epoch": best["epoch"], "best": best, "minutes": round((time.time() - t0) / 60, 1)}
    print(f"{name}: best epoch {best['epoch']} (val macro-F1 {best['val_macro_f1']:.3f}, val loss {best['val_loss']:.4f})")
    json.dump(RUNS, open(os.path.join(WORK, "training_metrics.json"), "w"), indent=1, default=str)

run_config("r16_lr1e-4", CFG_BASE)

## Cell 7 — Small, targeted search (validation only)
Two variants, each changing ONE thing from the main run: a smaller adapter (r = 8) and a lower learning rate
(5e-5). They run only if the time budget allows. Selection uses validation numbers only; the test split is never
looked at here.

In [ ]:
TIME_BUDGET_H = 8.0   # Kaggle GPU sessions stop at 12 h; keep a wide margin
VARIANTS = {"r8_lr1e-4": {**CFG_BASE, "r": 8, "alpha": 16}, "r16_lr5e-5": {**CFG_BASE, "lr": 5e-5}}
per_run_h = RUNS["r16_lr1e-4"]["minutes"] / 60
for name, cfg in VARIANTS.items():
    used_h = (time.time() - T_START) / 3600
    if used_h + per_run_h * 1.2 > TIME_BUDGET_H:
        print(f"skip {name}: {used_h:.1f} h used, about {per_run_h:.1f} h needed")
        continue
    del model; torch.cuda.empty_cache()
    model = make_peft(cfg)
    run_config(name, cfg)
BEST_RUN = max(RUNS, key=lambda n: (RUNS[n]["best"]["val_macro_f1"], -RUNS[n]["best"]["val_loss"]))
SEL = RUNS[BEST_RUN]
print("selected on validation:", BEST_RUN, "epoch", SEL["best_epoch"], "| val macro-F1", round(SEL["best"]["val_macro_f1"], 3))
for n, r in RUNS.items():
    print(f"  {n:12s} " + "  ".join(f"ep{h['epoch']}: loss {h['val_loss']:.3f} F1 {h['val_macro_f1']:.3f}" for h in r["history"]))

## Cell 8 — Final benchmark on the held-out test split
Same 25 test photos, same decoding, same scoring for: (1) original model + original prompt, (2) original model +
improved prompt, (3) fine-tuned adapter + improved prompt. 95 % bootstrap intervals are shown because the test
split is small; the paired bootstrap tells how sure we are that (3) beats (2).

In [ ]:
import csv
from peft import PeftModel
del model; torch.cuda.empty_cache()
# load exactly as CallVerse will: base model + saved adapter folder (checks that the checkpoint loads)
model = PeftModel.from_pretrained(load_base(COMPUTE), SEL["best"]["adapter_dir"])
model.eval()
model.config.use_cache = True
print("fine-tuned test predictions:")
PRED[("finetuned_qlora", "test")] = L.predict(model, processor, split["test"], L.PROMPT_IMPROVED)
PRED[("finetuned_qlora", "val")] = L.predict(model, processor, split["val"], L.PROMPT_IMPROVED)
CHECK["val8_loss_true_images_after"] = L.eval_loss(model, processor, probe_rows, L.PROMPT_IMPROVED)
CHECK["val8_loss_swapped_images_after"] = L.eval_loss(model, processor, probe_rows, L.PROMPT_IMPROVED, shuffle_images=True)
print("image check (lower loss with the true photo = the adapter uses the image):", json.dumps(CHECK, indent=1))

SYSTEMS = ["base_original_prompt", "base_improved_prompt", "finetuned_qlora"]
RES = {(sy, s): L.metrics(split[s], PRED[(sy, s)]) for sy in SYSTEMS for s in ("val", "test")}
with open("benchmark_comparison.csv", "w", newline="") as f:
    w = csv.writer(f)
    w.writerow(["system", "split", "n", "accuracy", "accuracy_ci95", "macro_precision", "macro_recall", "macro_f1",
                "macro_f1_ci95", "damage_recall", "damage_recall_ci95", "damage_precision", "false_alarm_rate",
                "damage_recall_lenient", "json_validity", "latency_mean_s", "latency_median_s", "latency_p95_s"])
    for (sy, s), m in RES.items():
        w.writerow([sy, s, m["n"], round(m["accuracy"], 4), m["ci95"]["accuracy"], round(m["macro_precision"], 4),
                    round(m["macro_recall"], 4), round(m["macro_f1"], 4), m["ci95"]["macro_f1"], round(m["damage_recall"], 4),
                    m["ci95"]["damage_recall"], round(m["damage_precision"], 4), round(m["false_alarm_rate"], 4),
                    round(m["damage_recall_lenient"], 4), round(m["json_validity"], 4), round(m["latency_s"]["mean"], 3), round(m["latency_s"]["median"], 3),
                    round(m["latency_s"]["p95"], 3)])
print(open("benchmark_comparison.csv").read())
print("per-class F1 on test:")
for c in sorted({r["target"]["issue_type"] for r in split["test"]}):
    print(f"  {c:24s} " + "  ".join(f"{sy}: {RES[(sy,'test')]['per_class'][c]['f1']:.2f}" for sy in SYSTEMS))
PAIRED = {"finetuned_vs_improved_prompt": L.paired_bootstrap(split["test"], PRED[("base_improved_prompt", "test")], PRED[("finetuned_qlora", "test")]),
          "finetuned_vs_original": L.paired_bootstrap(split["test"], PRED[("base_original_prompt", "test")], PRED[("finetuned_qlora", "test")])}
print("paired bootstrap on test accuracy:", json.dumps(PAIRED, indent=1))

# confusion matrices (rows = truth, columns = prediction, last column = invalid output)
cols = L.ISSUE_TYPES + ["INVALID"]
fig, axes = plt.subplots(1, 3, figsize=(24, 7))
for ax, sy in zip(axes, SYSTEMS):
    M = np.zeros((len(L.ISSUE_TYPES), len(cols)), dtype=int)
    for r, p in zip(split["test"], PRED[(sy, "test")]):
        M[L.ISSUE_TYPES.index(r["target"]["issue_type"]), cols.index(p["parsed"]["issue_type"] if p["parsed"] else "INVALID")] += 1
    ax.imshow(M, cmap="Blues")
    ax.set_xticks(range(len(cols))); ax.set_xticklabels(cols, rotation=60, ha="right", fontsize=8)
    ax.set_yticks(range(len(L.ISSUE_TYPES))); ax.set_yticklabels(L.ISSUE_TYPES, fontsize=8)
    for i in range(M.shape[0]):
        for j in range(M.shape[1]):
            if M[i, j]:
                ax.text(j, i, M[i, j], ha="center", va="center", fontsize=9)
    ax.set_title(f"{sy} (test, n={len(split['test'])})")
plt.tight_layout(); plt.savefig("confusion_matrix.png", dpi=90); plt.show()

with open("error_analysis.csv", "w", newline="") as f:
    w = csv.writer(f)
    w.writerow(["system", "split", "id", "domain", "gold_issue", "pred_issue", "gold_damage", "pred_damage", "parse",
                "correct", "gold_description", "raw_output"])
    for sy in SYSTEMS:
        for s in ("val", "test"):
            for r, p in zip(split[s], PRED[(sy, s)]):
                pi = p["parsed"]["issue_type"] if p["parsed"] else None
                w.writerow([sy, s, r["id"], r["domain"], r["target"]["issue_type"], pi, r["target"]["visible_damage"],
                            p["parsed"]["visible_damage"] if p["parsed"] else None, p["parse"], pi == r["target"]["issue_type"],
                            r["target"]["description"], p["raw"]])
with open("predictions.jsonl", "w") as f:
    for (sy, s), ps in PRED.items():
        for p in ps:
            f.write(json.dumps({"system": sy, "split": s, **p}, ensure_ascii=False) + "\n")

## Cell 9 — Save the best adapter

In [ ]:
import shutil
BEST_DIR = os.path.join(WORK, "best_adapter")
shutil.rmtree(BEST_DIR, ignore_errors=True)
shutil.copytree(SEL["best"]["adapter_dir"], BEST_DIR)
processor.save_pretrained(BEST_DIR)          # same image size settings as in training
info = {"base_model": BASE_MODEL, "base_revision": BASE_REVISION, "run": BEST_RUN, "epoch": SEL["best_epoch"],
        "quantisation_used_in_training": QUANT, "compute_dtype": str(COMPUTE), "max_pixels": MAX_PIXELS,
        "min_pixels": MIN_PIXELS, "prompt": L.PROMPT_IMPROVED, "output_keys": L.KEYS, "issue_types": L.ISSUE_TYPES,
        "selected_on": "validation macro-F1 (tie: validation loss)", "load_note":
        "Load Qwen3VLForConditionalGeneration(BASE_MODEL, revision=BASE_REVISION) in 4-bit NF4 (or 16-bit), then "
        "PeftModel.from_pretrained(model, best_adapter). Use the prompt above and greedy decoding."}
json.dump(info, open(os.path.join(BEST_DIR, "callverse_adapter_info.json"), "w"), indent=1)
print(os.listdir(BEST_DIR), round(sum(os.path.getsize(os.path.join(BEST_DIR, f)) for f in os.listdir(BEST_DIR)) / 2**20, 1), "MB")

## Cell 10 — Package all results

In [ ]:
import zipfile
json.dump({**CFG_BASE, "prompt": L.PROMPT_IMPROVED, "runs": {n: r["config"] for n, r in RUNS.items()},
           "selected_run": BEST_RUN, "selected_epoch": SEL["best_epoch"], "seed": SEED,
           "base_model": BASE_MODEL, "base_revision": BASE_REVISION}, open("training_config.json", "w"), indent=1, default=str)
TM = {"runs": RUNS, "selected_run": BEST_RUN, "image_conditioning_check": CHECK, "paired_bootstrap_test": PAIRED,
      "metrics": {f"{sy}|{s}": m for (sy, s), m in RES.items()}, "baseline_val": BASE_VAL}
json.dump(TM, open("training_metrics.json", "w"), indent=1, default=str)
env["total_runtime_min"] = round((time.time() - T_START) / 60, 1)
json.dump(env, open("environment_info.json", "w"), indent=1)
shutil.copy(os.path.join(DATA, "dataset.jsonl"), "dataset.jsonl")
if os.path.exists(os.path.join(DATA, "review_queue.csv")):
    shutil.copy(os.path.join(DATA, "review_queue.csv"), "review_queue.csv")
FILES = ["training_metrics.json", "benchmark_comparison.csv", "confusion_matrix.png", "error_analysis.csv",
         "dataset_summary.json", "training_config.json", "environment_info.json", "predictions.jsonl",
         "dataset.jsonl", "review_queue.csv", "sample_images.png"]
ZIP = os.path.join(WORK, "callverse_vision_qlora_results.zip")
with zipfile.ZipFile(ZIP, "w", zipfile.ZIP_DEFLATED) as z:
    for f in FILES:
        if os.path.exists(f):
            z.write(f)
    for f in os.listdir(BEST_DIR):
        z.write(os.path.join(BEST_DIR, f), os.path.join("best_adapter", f))
mb = os.path.getsize(ZIP) / 2**20
print(f"{ZIP}: {mb:.1f} MB")
if mb > 500:
    print("ZIP is large: download best_adapter/ separately from the Output panel (Kaggle keeps /kaggle/working).")
# keep the output small: the per-epoch checkpoints and the photo cache are not needed after packaging
shutil.rmtree(os.path.join(WORK, "source_cache"), ignore_errors=True)
print("total runtime:", env["total_runtime_min"], "min")